# Study the laplacian regression on the Larged Uncrompressed Graph

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import networkx as nx

from scipy import sparse
from scipy.sparse.linalg import spsolve

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 1. FILE LOCATIONS
# ============================================================

BASE_DIR = Path.cwd()

FULL_GRAPH_FILE = (
    BASE_DIR
    / "largest_river_component.pkl"
)

MASTER_GRAPH_FILE = (
    BASE_DIR
    / "master_compressed_river_graph.pkl"
)

HOURLY_FILE = (
    BASE_DIR
    / "usgs_all_stations_hourly.csv"
)


print("Current directory:")
print(BASE_DIR)


# ============================================================
# 2. CHECK FILES EXIST
# ============================================================

required_files = [
    FULL_GRAPH_FILE,
    MASTER_GRAPH_FILE,
    HOURLY_FILE
]


for file in required_files:

    print(
        file.name,
        "->",
        "FOUND"
        if file.exists()
        else "MISSING"
    )


missing_files = [
    file
    for file in required_files
    if not file.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Missing files:\n"
        +
        "\n".join(
            str(file)
            for file in missing_files
        )
    )


# ============================================================
# 3. LOAD UNCOMPRESSED LARGEST COMPONENT
# ============================================================

with open(
    FULL_GRAPH_FILE,
    "rb"
) as f:

    G_full = pickle.load(f)


print("\n" + "=" * 60)
print("FULL / UNCOMPRESSED GRAPH")
print("=" * 60)

print(
    "Directed:",
    G_full.is_directed()
)

print(
    "Nodes:",
    G_full.number_of_nodes()
)

print(
    "Edges:",
    G_full.number_of_edges()
)


if G_full.is_directed():

    print(
        "Weak components:",
        nx.number_weakly_connected_components(
            G_full
        )
    )

else:

    print(
        "Connected components:",
        nx.number_connected_components(
            G_full
        )
    )


# ============================================================
# 4. LOAD MASTER COMPRESSED GRAPH
#
# We are NOT using this for the new Laplacian.
# We only need its station/support metadata.
# ============================================================

with open(
    MASTER_GRAPH_FILE,
    "rb"
) as f:

    G_master = pickle.load(f)


print("\n" + "=" * 60)
print("MASTER METADATA GRAPH")
print("=" * 60)

print(
    "Nodes:",
    G_master.number_of_nodes()
)

print(
    "Edges:",
    G_master.number_of_edges()
)


# ============================================================
# 5. LOAD HOURLY USGS DATA
# ============================================================

hourly = pd.read_csv(
    HOURLY_FILE
)


hourly["time"] = pd.to_datetime(
    hourly["time"],
    utc=True,
    errors="coerce"
)


print("\n" + "=" * 60)
print("HOURLY DATA")
print("=" * 60)

print(
    "Rows:",
    len(hourly)
)

print(
    "Stations:",
    hourly["station_id"].nunique()
)

print(
    "Columns:"
)

print(
    hourly.columns.tolist()
)


# ============================================================
# 6. BUILD STATION -> GRAPH NODE MAPPING
# ============================================================

station_to_node = {}


for node, data in G_master.nodes(
    data=True
):

    clusters = data.get(
        "usgs_clusters",
        []
    )


    for cluster in clusters:

        stations = cluster.get(
            "member_stations",
            []
        )


        # fallback
        if not stations:

            anchor = cluster.get(
                "anchor_station"
            )

            if anchor is not None:

                stations = [
                    anchor
                ]


        for station in stations:

            station_to_node[
                station
            ] = node


print(
    "\nWQ stations mapped:",
    len(station_to_node)
)


# ============================================================
# 7. IDENTIFY WQ NODES
# ============================================================

all_wq_nodes = {

    node

    for node, data
    in G_master.nodes(
        data=True
    )

    if (
        data.get(
            "usgs_station_count",
            0
        ) > 0

        or

        len(
            data.get(
                "usgs_clusters",
                []
            )
        ) > 0
    )
}


# ============================================================
# 8. IDENTIFY SUPPORT-GAUGE NODES
# ============================================================

support_nodes = {

    node

    for node, data
    in G_master.nodes(
        data=True
    )

    if (
        data.get(
            "support_gauge_count",
            0
        ) > 0

        or

        len(
            data.get(
                "support_gauges",
                []
            )
        ) > 0
    )
}


print(
    "WQ graph nodes:",
    len(all_wq_nodes)
)

print(
    "Support-gauge nodes:",
    len(support_nodes)
)


# ============================================================
# 9. VERIFY THEY EXIST IN FULL GRAPH
# ============================================================

full_nodes = set(
    G_full.nodes()
)


missing_wq = (
    all_wq_nodes
    - full_nodes
)

missing_support = (
    support_nodes
    - full_nodes
)


print("\n" + "=" * 60)
print("FULL-GRAPH MONITORING CHECK")
print("=" * 60)


print(
    "WQ nodes present:",
    len(
        all_wq_nodes
        &
        full_nodes
    ),
    "/",
    len(all_wq_nodes)
)


print(
    "Support nodes present:",
    len(
        support_nodes
        &
        full_nodes
    ),
    "/",
    len(support_nodes)
)


print(
    "\nMissing WQ nodes:",
    sorted(
        missing_wq
    )
)

print(
    "Missing support nodes:",
    sorted(
        missing_support
    )
)


# ============================================================
# 10. ATTACH GRAPH NODE TO HOURLY DATA
# ============================================================

hourly["node"] = (
    hourly["station_id"]
    .map(
        station_to_node
    )
)


mapped_hourly = (
    hourly[
        hourly["node"]
        .notna()
    ]
    .copy()
)


mapped_hourly[
    "node"
] = (
    mapped_hourly[
        "node"
    ]
    .astype(int)
)


print("\n" + "=" * 60)
print("READY FOR MODELING")
print("=" * 60)

print(
    "Mapped hourly rows:",
    len(mapped_hourly)
)

print(
    "Mapped WQ graph nodes:",
    mapped_hourly[
        "node"
    ].nunique()
)

print(
    "Full graph nodes available to Laplacian:",
    G_full.number_of_nodes()
)

Current directory:
/Users/andrewchincea/TDA_Lab_Practice/rivernets/making_the_graph
largest_river_component.pkl -> FOUND
master_compressed_river_graph.pkl -> FOUND
usgs_all_stations_hourly.csv -> FOUND

FULL / UNCOMPRESSED GRAPH
Directed: True
Nodes: 67401
Edges: 67400
Weak components: 1

MASTER METADATA GRAPH
Nodes: 67
Edges: 66

HOURLY DATA
Rows: 234095
Stations: 28
Columns:
['station_id', 'time', 'DO concentration', 'Water temperature', 'Specific conductance', 'Turbidity', 'pH', 'Discharge', 'Stage']

WQ stations mapped: 28
WQ graph nodes: 26
Support-gauge nodes: 3

FULL-GRAPH MONITORING CHECK
WQ nodes present: 26 / 26
Support nodes present: 3 / 3

Missing WQ nodes: []
Missing support nodes: []

READY FOR MODELING
Mapped hourly rows: 234095
Mapped WQ graph nodes: 26
Full graph nodes available to Laplacian: 67401


In [2]:
# ============================================================
# Graph:
#     67,401-node uncompressed largest component
#
# Edge weight:
#     w_ij = 1 / distance_km
#
# Standard undirected combinatorial Laplacian:
#     L = D - W
# ============================================================

from scipy import sparse
from math import radians, sin, cos, asin, sqrt


# ============================================================
# 1. HAVERSINE DISTANCE
# ============================================================

def haversine_m(
    lon1,
    lat1,
    lon2,
    lat2
):

    R = 6371000.0

    lon1 = radians(float(lon1))
    lat1 = radians(float(lat1))
    lon2 = radians(float(lon2))
    lat2 = radians(float(lat2))

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        sin(dlat / 2.0) ** 2
        +
        cos(lat1)
        * cos(lat2)
        * sin(dlon / 2.0) ** 2
    )

    return (
        2.0
        * R
        * asin(
            sqrt(a)
        )
    )


# ============================================================
# 2. FIXED NODE ORDER
# ============================================================

full_node_order = list(
    G_full.nodes()
)

full_node_index = {
    node: i
    for i, node
    in enumerate(full_node_order)
}

N_FULL = len(
    full_node_order
)

print(
    "Full graph node count:",
    N_FULL
)


# ============================================================
# 3. BUILD SPARSE ADJACENCY
# ============================================================

rows = []
cols = []
weights = []

edge_distances_km = []


# Treat directed river graph as undirected for standard
# symmetric Graph-Laplacian regression.
G_full_und = G_full.to_undirected(
    as_view=True
)


for u, v, data in G_full_und.edges(
    data=True
):

    # --------------------------------------------------------
    # Physical edge distance
    # --------------------------------------------------------

    length_m = data.get(
        "length_m",
        None
    )


    if (
        length_m is None
        or not np.isfinite(
            float(length_m)
        )
        or float(length_m) <= 0
    ):

        udata = G_full.nodes[u]
        vdata = G_full.nodes[v]

        length_m = haversine_m(

            udata["longitude"],
            udata["latitude"],

            vdata["longitude"],
            vdata["latitude"]
        )


    distance_km = max(
        float(length_m) / 1000.0,
        1e-8
    )


    edge_distances_km.append(
        distance_km
    )


    # --------------------------------------------------------
    # Conductance-style edge weight
    # --------------------------------------------------------

    w = (
        1.0
        /
        distance_km
    )


    i = full_node_index[u]
    j = full_node_index[v]


    rows.extend(
        [
            i,
            j
        ]
    )

    cols.extend(
        [
            j,
            i
        ]
    )

    weights.extend(
        [
            w,
            w
        ]
    )


# ============================================================
# 4. SPARSE W
# ============================================================

W_full = sparse.coo_matrix(

    (
        weights,
        (
            rows,
            cols
        )
    ),

    shape=(
        N_FULL,
        N_FULL
    )

).tocsr()


# ============================================================
# 5. SPARSE LAPLACIAN
# ============================================================

degree = np.asarray(
    W_full.sum(
        axis=1
    )
).ravel()


L_full = (
    sparse.diags(
        degree
    )
    -
    W_full
).tocsr()


# ============================================================
# 6. CHECKS
# ============================================================

edge_distances_km = np.asarray(
    edge_distances_km
)


print(
    "\nLaplacian shape:",
    L_full.shape
)

print(
    "Nonzero entries:",
    L_full.nnz
)


storage_mb = (

    L_full.data.nbytes
    +
    L_full.indices.nbytes
    +
    L_full.indptr.nbytes

) / 1024**2


print(
    "Approx sparse storage:",
    round(storage_mb, 2),
    "MB"
)


print(
    "\nEdge distance km:"
)

print(
    pd.Series(
        edge_distances_km
    ).describe()
)


symmetry_error = (
    L_full
    -
    L_full.T
)


print(
    "\nSymmetric:",
    symmetry_error.nnz == 0
)


print(
    "Finite:",
    np.isfinite(
        L_full.data
    ).all()
)


# Row sums should be approximately zero
row_sums = np.asarray(
    L_full.sum(
        axis=1
    )
).ravel()


print(
    "Maximum absolute row sum:",
    np.abs(
        row_sums
    ).max()
)

Full graph node count: 67401

Laplacian shape: (67401, 67401)
Nonzero entries: 202201
Approx sparse storage: 2.57 MB

Edge distance km:
count    67400.000000
mean         0.097055
std          0.019850
min          0.068725
25%          0.076338
50%          0.092662
75%          0.118483
max          0.121598
dtype: float64

Symmetric: True
Finite: True
Maximum absolute row sum: 7.105427357601002e-15


In [3]:
from itertools import combinations


WQ_VARIABLES = [
    "DO concentration",
    "Water temperature",
    "Specific conductance",
    "Turbidity",
    "pH"
]


# ============================================================
# 1. COLLAPSE TO ONE ROW PER GRAPH NODE / HOUR
# ============================================================

node_hourly_all = (
    mapped_hourly[
        [
            "node",
            "time"
        ]
        + WQ_VARIABLES
    ]
    .groupby(
        [
            "node",
            "time"
        ],
        as_index=False
    )[
        WQ_VARIABLES
    ]
    .median()
)


print(
    "Node-hour rows:",
    len(node_hourly_all)
)

print(
    "WQ graph nodes represented:",
    node_hourly_all[
        "node"
    ].nunique()
)


# ============================================================
# 2. TEST EVERY PREDICTOR SUBSET FOR EVERY TARGET
# ============================================================

coverage_rows = []


for target in WQ_VARIABLES:

    candidate_features = [
        variable
        for variable in WQ_VARIABLES
        if variable != target
    ]


    # Every node that has the target at least once
    target_nodes = set(
        node_hourly_all.loc[
            node_hourly_all[
                target
            ].notna(),
            "node"
        ].unique()
    )


    for r in range(
        1,
        len(candidate_features) + 1
    ):

        for combo in combinations(
            candidate_features,
            r
        ):

            features = list(
                combo
            )


            usable = (
                node_hourly_all
                .dropna(
                    subset=[
                        target
                    ] + features
                )
            )


            counts = (
                usable
                .groupby(
                    "node"
                )
                .size()
            )


            usable_nodes = set(
                counts.index
            )


            missing_nodes = sorted(
                target_nodes
                -
                usable_nodes
            )


            coverage_rows.append({

                "target":
                    target,

                "features":
                    " + ".join(
                        features
                    ),

                "n_features":
                    len(
                        features
                    ),

                "target_nodes":
                    len(
                        target_nodes
                    ),

                "usable_nodes":
                    len(
                        usable_nodes
                    ),

                "all_target_nodes_covered":
                    (
                        usable_nodes
                        ==
                        target_nodes
                    ),

                "total_node_hours":
                    len(
                        usable
                    ),

                "minimum_hours_per_node":
                    (
                        int(
                            counts.min()
                        )
                        if len(counts)
                        else 0
                    ),

                "missing_nodes":
                    missing_nodes
            })


coverage_table = pd.DataFrame(
    coverage_rows
)


# ============================================================
# 3. DISPLAY BEST OPTIONS FOR EACH TARGET
# ============================================================

for target in WQ_VARIABLES:

    print(
        "\n"
        + "=" * 75
    )

    print(
        "TARGET:",
        target
    )

    print(
        "=" * 75
    )


    target_table = (
        coverage_table[
            coverage_table[
                "target"
            ] == target
        ]
        .sort_values(
            [
                "all_target_nodes_covered",
                "usable_nodes",
                "n_features",
                "total_node_hours"
            ],
            ascending=[
                False,
                False,
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )


    display(
        target_table.head(
            12
        )
    )

Node-hour rows: 225339
WQ graph nodes represented: 26

TARGET: DO concentration


,target,features,n_features,target_nodes,usable_nodes,all_target_nodes_covered,total_node_hours,minimum_hours_per_node,missing_nodes
0,DO concentration,Water temperature + Specific conductance,2,17,17,True,114607,1885,[]
1,DO concentration,Water temperature,1,17,17,True,115353,1885,[]
2,DO concentration,Specific conductance,1,17,17,True,114608,1885,[]
3,DO concentration,Water temperature + Specific conductance + pH,3,17,16,False,109620,4113,[92116891]
4,DO concentration,Water temperature + pH,2,17,16,False,110231,4140,[92116891]
5,DO concentration,Specific conductance + pH,2,17,16,False,109621,4113,[92116891]
6,DO concentration,pH,1,17,16,False,110232,4140,[92116891]
7,DO concentration,Water temperature + Specific conductance + Tur...,4,17,14,False,80592,1140,"[92116891, 186559442, 203177183]"
8,DO concentration,Water temperature + Specific conductance + Tur...,3,17,14,False,82418,1140,"[92116891, 186559442, 203177183]"
9,DO concentration,Water temperature + Turbidity + pH,3,17,14,False,80962,1140,"[92116891, 186559442, 203177183]"



TARGET: Water temperature


,target,features,n_features,target_nodes,usable_nodes,all_target_nodes_covered,total_node_hours,minimum_hours_per_node,missing_nodes
0,Water temperature,Specific conductance,1,25,24,False,144980,168,[129078768]
1,Water temperature,Specific conductance + pH,2,25,18,False,120839,168,"[47787846, 58738608, 75465016, 92116891, 96751..."
2,Water temperature,pH,1,25,18,False,121821,168,"[47787846, 58738608, 75465016, 92116891, 96751..."
3,Water temperature,DO concentration + Specific conductance,2,25,17,False,114607,1885,"[47787846, 58738608, 75465016, 96751415, 11091..."
4,Water temperature,DO concentration,1,25,17,False,115353,1885,"[47787846, 58738608, 75465016, 96751415, 11091..."
5,Water temperature,DO concentration + Specific conductance + pH,3,25,16,False,109620,4113,"[47787846, 58738608, 75465016, 92116891, 96751..."
6,Water temperature,Specific conductance + Turbidity + pH,3,25,16,False,87966,134,"[47787846, 58738608, 75465016, 92116891, 96751..."
7,Water temperature,DO concentration + pH,2,25,16,False,110231,4140,"[47787846, 58738608, 75465016, 92116891, 96751..."
8,Water temperature,Specific conductance + Turbidity,2,25,16,False,89948,134,"[47787846, 58738608, 75465016, 92116891, 96751..."
9,Water temperature,Turbidity + pH,2,25,16,False,88666,134,"[47787846, 58738608, 75465016, 92116891, 96751..."



TARGET: Specific conductance


,target,features,n_features,target_nodes,usable_nodes,all_target_nodes_covered,total_node_hours,minimum_hours_per_node,missing_nodes
0,Specific conductance,Water temperature,1,25,24,False,144980,168,[129078768]
1,Specific conductance,pH,1,25,19,False,121011,168,"[47787846, 58738608, 75465016, 92116891, 96751..."
2,Specific conductance,Water temperature + pH,2,25,18,False,120839,168,"[47787846, 58738608, 75465016, 92116891, 96751..."
3,Specific conductance,DO concentration + Water temperature,2,25,17,False,114607,1885,"[47787846, 58738608, 75465016, 96751415, 11091..."
4,Specific conductance,Turbidity + pH,2,25,17,False,88095,126,"[47787846, 58738608, 75465016, 92116891, 96751..."
5,Specific conductance,DO concentration,1,25,17,False,114608,1885,"[47787846, 58738608, 75465016, 96751415, 11091..."
6,Specific conductance,Turbidity,1,25,17,False,90077,126,"[47787846, 58738608, 75465016, 92116891, 96751..."
7,Specific conductance,DO concentration + Water temperature + pH,3,25,16,False,109620,4113,"[47787846, 58738608, 75465016, 92116891, 96751..."
8,Specific conductance,Water temperature + Turbidity + pH,3,25,16,False,87966,134,"[47787846, 58738608, 75465016, 92116891, 96751..."
9,Specific conductance,DO concentration + pH,2,25,16,False,109621,4113,"[47787846, 58738608, 75465016, 92116891, 96751..."



TARGET: Turbidity


,target,features,n_features,target_nodes,usable_nodes,all_target_nodes_covered,total_node_hours,minimum_hours_per_node,missing_nodes
0,Turbidity,Specific conductance + pH,2,17,17,True,88095,126,[]
1,Turbidity,Specific conductance,1,17,17,True,90077,126,[]
2,Turbidity,pH,1,17,17,True,88833,126,[]
3,Turbidity,Water temperature + Specific conductance + pH,3,17,16,False,87966,134,[129078768]
4,Turbidity,Water temperature + Specific conductance,2,17,16,False,89948,134,[129078768]
5,Turbidity,Water temperature + pH,2,17,16,False,88666,134,[129078768]
6,Turbidity,Water temperature,1,17,16,False,90672,134,[129078768]
7,Turbidity,DO concentration + Water temperature + Specifi...,4,17,14,False,80592,1140,"[129078768, 133114453, 134075067]"
8,Turbidity,DO concentration + Water temperature + Specifi...,3,17,14,False,82418,1140,"[129078768, 133114453, 134075067]"
9,Turbidity,DO concentration + Water temperature + pH,3,17,14,False,80962,1140,"[129078768, 133114453, 134075067]"



TARGET: pH


,target,features,n_features,target_nodes,usable_nodes,all_target_nodes_covered,total_node_hours,minimum_hours_per_node,missing_nodes
0,pH,Specific conductance,1,19,19,True,121011,168,[]
1,pH,Water temperature + Specific conductance,2,19,18,False,120839,168,[129078768]
2,pH,Water temperature,1,19,18,False,121821,168,[129078768]
3,pH,Specific conductance + Turbidity,2,19,17,False,88095,126,"[186559442, 203177183]"
4,pH,Turbidity,1,19,17,False,88833,126,"[186559442, 203177183]"
5,pH,DO concentration + Water temperature + Specifi...,3,19,16,False,109620,4113,"[129078768, 133114453, 134075067]"
6,pH,Water temperature + Specific conductance + Tur...,3,19,16,False,87966,134,"[129078768, 186559442, 203177183]"
7,pH,DO concentration + Water temperature,2,19,16,False,110231,4140,"[129078768, 133114453, 134075067]"
8,pH,DO concentration + Specific conductance,2,19,16,False,109621,4113,"[129078768, 133114453, 134075067]"
9,pH,Water temperature + Turbidity,2,19,16,False,88666,134,"[129078768, 186559442, 203177183]"


In [4]:
best_feature_rows = []


for target in WQ_VARIABLES:

    target_table = (
        coverage_table[
            coverage_table[
                "target"
            ] == target
        ]
        .copy()
    )


    full_coverage = (
        target_table[
            target_table[
                "all_target_nodes_covered"
            ]
        ]
        .sort_values(
            [
                "n_features",
                "total_node_hours",
                "minimum_hours_per_node"
            ],
            ascending=[
                False,
                False,
                False
            ]
        )
    )


    if len(
        full_coverage
    ) > 0:

        best = (
            full_coverage
            .iloc[0]
        )

        status = (
            "ALL target nodes"
        )

    else:

        # No predictor set reaches every target node.
        # Show maximum possible coverage instead.
        best = (
            target_table
            .sort_values(
                [
                    "usable_nodes",
                    "n_features",
                    "total_node_hours"
                ],
                ascending=[
                    False,
                    False,
                    False
                ]
            )
            .iloc[0]
        )

        status = (
            "Maximum available coverage"
        )


    best_feature_rows.append({

        "target":
            target,

        "target_nodes":
            best[
                "target_nodes"
            ],

        "usable_nodes":
            best[
                "usable_nodes"
            ],

        "predictors":
            best[
                "features"
            ],

        "n_predictors":
            best[
                "n_features"
            ],

        "node_hours":
            best[
                "total_node_hours"
            ],

        "minimum_hours_per_node":
            best[
                "minimum_hours_per_node"
            ],

        "coverage_status":
            status
    })


best_feature_summary = (
    pd.DataFrame(
        best_feature_rows
    )
)


display(
    best_feature_summary
)

,target,target_nodes,usable_nodes,predictors,n_predictors,node_hours,minimum_hours_per_node,coverage_status
0,DO concentration,17,17,Water temperature + Specific conductance,2,114607,1885,ALL target nodes
1,Water temperature,25,24,Specific conductance,1,144980,168,Maximum available coverage
2,Specific conductance,25,24,Water temperature,1,144980,168,Maximum available coverage
3,Turbidity,17,17,Specific conductance + pH,2,88095,126,ALL target nodes
4,pH,19,19,Specific conductance,1,121011,168,ALL target nodes


In [5]:
import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse.linalg import splu


boundary_nodes = sorted(
    all_wq_nodes
)

boundary_index = {
    node: i
    for i, node in enumerate(boundary_nodes)
}

N_BOUNDARY = len(
    boundary_nodes
)


print(
    "Boundary WQ nodes:",
    N_BOUNDARY
)


boundary_full_idx = np.array(
    [
        full_node_index[node]
        for node in boundary_nodes
    ],
    dtype=int
)


all_full_idx = np.arange(
    len(full_node_order)
)


interior_mask = np.ones(
    len(full_node_order),
    dtype=bool
)

interior_mask[
    boundary_full_idx
] = False


interior_full_idx = all_full_idx[
    interior_mask
]


print(
    "Interior/unlabeled nodes:",
    len(interior_full_idx)
)


L_BB = (
    L_full[
        boundary_full_idx,
        :
    ][
        :,
        boundary_full_idx
    ]
    .tocsc()
)


L_BI = (
    L_full[
        boundary_full_idx,
        :
    ][
        :,
        interior_full_idx
    ]
    .tocsc()
)


L_IB = (
    L_full[
        interior_full_idx,
        :
    ][
        :,
        boundary_full_idx
    ]
    .tocsc()
)


L_II = (
    L_full[
        interior_full_idx,
        :
    ][
        :,
        interior_full_idx
    ]
    .tocsc()
)


print(
    "L_BB:",
    L_BB.shape
)

print(
    "L_II:",
    L_II.shape
)


print(
    "\nFactorizing interior Laplacian..."
)

lu = splu(
    L_II
)

print(
    "Factorization complete."
)


rhs = (
    L_IB
    .toarray()
)


solution = lu.solve(
    rhs
)


L_kron = (
    L_BB.toarray()
    -
    L_BI @ solution
)


L_kron = np.asarray(
    L_kron
)


L_kron = (
    L_kron
    +
    L_kron.T
) / 2.0


for i in range(
    N_BOUNDARY
):

    off_diag_sum = (
        L_kron[i, :].sum()
        -
        L_kron[i, i]
    )

    L_kron[
        i,
        i
    ] = -off_diag_sum


positive_diag = np.diag(
    L_kron
)

positive_diag = positive_diag[
    positive_diag > 1e-12
]


LAPLACIAN_SCALE = (
    positive_diag.mean()
)


L_kron_scaled = (
    L_kron
    /
    LAPLACIAN_SCALE
)


print(
    "\nReduced Laplacian shape:",
    L_kron_scaled.shape
)

print(
    "Scale factor:",
    LAPLACIAN_SCALE
)

print(
    "Symmetry error:",
    np.max(
        np.abs(
            L_kron_scaled
            -
            L_kron_scaled.T
        )
    )
)

print(
    "Maximum row-sum error:",
    np.max(
        np.abs(
            L_kron_scaled.sum(
                axis=1
            )
        )
    )
)


eigenvalues = np.linalg.eigvalsh(
    L_kron_scaled
)


print(
    "Smallest eigenvalues:"
)

print(
    eigenvalues[:5]
)


print(
    "\nKron reduction complete."
)

print(
    "The model will use the full 67k-node network"
)

print(
    "through this exact 26-node effective Laplacian."
)

Boundary WQ nodes: 26
Interior/unlabeled nodes: 67375
L_BB: (26, 26)
L_II: (67375, 67375)

Factorizing interior Laplacian...
Factorization complete.

Reduced Laplacian shape: (26, 26)
Scale factor: 0.047553741042819814
Symmetry error: 0.0
Maximum row-sum error: 2.220446049250313e-16
Smallest eigenvalues:
[-2.75278931e-16  5.43830838e-03  8.04522277e-03  1.10802187e-02
  4.11998132e-02]

Kron reduction complete.
The model will use the full 67k-node network
through this exact 26-node effective Laplacian.


In [6]:
TARGET_FEATURES = {

    "DO concentration": [
        "Water temperature",
        "Specific conductance"
    ],

    "Water temperature": [
        "Specific conductance"
    ],

    "Specific conductance": [
        "Water temperature"
    ],

    "Turbidity": [
        "Specific conductance",
        "pH"
    ],

    "pH": [
        "Specific conductance"
    ]
}


target_datasets = {}


for target, features in TARGET_FEATURES.items():

    needed = (
        [
            "node",
            "time",
            target
        ]
        +
        features
    )


    target_df = (
        node_hourly_all[
            needed
        ]
        .dropna(
            subset=[
                target
            ] + features
        )
        .copy()
    )


    target_datasets[
        target
    ] = target_df


    print(
        "\n",
        target
    )

    print(
        "  predictors:",
        features
    )

    print(
        "  nodes:",
        target_df[
            "node"
        ].nunique()
    )

    print(
        "  node-hours:",
        len(
            target_df
        )
    )


 DO concentration
  predictors: ['Water temperature', 'Specific conductance']
  nodes: 17
  node-hours: 114607

 Water temperature
  predictors: ['Specific conductance']
  nodes: 24
  node-hours: 144980

 Specific conductance
  predictors: ['Water temperature']
  nodes: 24
  node-hours: 144980

 Turbidity
  predictors: ['Specific conductance', 'pH']
  nodes: 17
  node-hours: 88095

 pH
  predictors: ['Specific conductance']
  nodes: 19
  node-hours: 121011


In [7]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


LINEAR_ALPHA_GRID = [
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0
]


GRAPH_LAMBDA_GRID = [
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0,
    30.0,
    100.0
]


GRAPH_BETA_GRID = [
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0
]


def make_node_stats(
    df,
    target,
    features
):

    stats = {}


    for node, group in df.groupby(
        "node"
    ):

        X = (
            group[
                features
            ]
            .to_numpy(
                dtype=float
            )
        )

        y = (
            group[
                target
            ]
            .to_numpy(
                dtype=float
            )
        )


        n = len(
            group
        )


        stats[
            int(node)
        ] = {

            "n":
                n,

            "mean_x":
                X.mean(
                    axis=0
                ),

            "exx":
                (
                    X.T @ X
                ) / n,

            "exy":
                (
                    X.T @ y
                ) / n,

            "mean_y":
                y.mean(),

            "mean_y2":
                np.mean(
                    y ** 2
                )
        }


    return stats


def training_scaling(
    stats,
    train_nodes
):

    mean_x = np.mean(
        [
            stats[node][
                "mean_x"
            ]
            for node
            in train_nodes
        ],
        axis=0
    )


    ex2 = np.mean(
        [
            np.diag(
                stats[node][
                    "exx"
                ]
            )
            for node
            in train_nodes
        ],
        axis=0
    )


    variance = (
        ex2
        -
        mean_x ** 2
    )


    variance = np.maximum(
        variance,
        1e-12
    )


    std_x = np.sqrt(
        variance
    )


    return (
        mean_x,
        std_x
    )


def standard_moments(
    stat,
    mean_x,
    std_x
):

    mx = stat[
        "mean_x"
    ]

    exx = stat[
        "exx"
    ]

    exy = stat[
        "exy"
    ]

    my = stat[
        "mean_y"
    ]


    zmean = (
        mx - mean_x
    ) / std_x


    zexx = (

        exx

        - np.outer(
            mx,
            mean_x
        )

        - np.outer(
            mean_x,
            mx
        )

        + np.outer(
            mean_x,
            mean_x
        )

    ) / np.outer(
        std_x,
        std_x
    )


    zexy = (
        exy
        -
        mean_x * my
    ) / std_x


    qmean = np.concatenate(
        (
            [1.0],
            zmean
        )
    )


    p = len(
        mean_x
    )


    q2 = np.zeros(
        (
            p + 1,
            p + 1
        )
    )


    q2[
        0,
        0
    ] = 1.0


    q2[
        0,
        1:
    ] = zmean


    q2[
        1:,
        0
    ] = zmean


    q2[
        1:,
        1:
    ] = zexx


    qy = np.concatenate(
        (
            [my],
            zexy
        )
    )


    return (
        qmean,
        q2,
        qy
    )


def fit_linear_model(
    stats,
    train_nodes,
    alpha
):

    mean_x, std_x = (
        training_scaling(
            stats,
            train_nodes
        )
    )


    p = len(
        mean_x
    )


    A = np.zeros(
        (
            p + 1,
            p + 1
        )
    )


    b = np.zeros(
        p + 1
    )


    for node in train_nodes:

        _, q2, qy = (
            standard_moments(
                stats[node],
                mean_x,
                std_x
            )
        )


        A += q2
        b += qy


    penalty = np.eye(
        p + 1
    )

    penalty[
        0,
        0
    ] = 0.0


    beta = np.linalg.solve(

        A
        +
        alpha
        * penalty,

        b
    )


    return {

        "beta":
            beta,

        "mean_x":
            mean_x,

        "std_x":
            std_x
    }


def fit_graph_model(
    stats,
    train_nodes,
    lambda_lap,
    lambda_beta
):

    mean_x, std_x = (
        training_scaling(
            stats,
            train_nodes
        )
    )


    p = len(
        mean_x
    )


    A11 = np.zeros(
        (
            p + 1,
            p + 1
        )
    )


    A12 = np.zeros(
        (
            p + 1,
            N_BOUNDARY
        )
    )


    rhs_beta = np.zeros(
        p + 1
    )


    mask = np.zeros(
        N_BOUNDARY
    )


    rhs_g = np.zeros(
        N_BOUNDARY
    )


    for node in train_nodes:

        qmean, q2, qy = (
            standard_moments(
                stats[node],
                mean_x,
                std_x
            )
        )


        j = boundary_index[
            node
        ]


        A11 += q2

        A12[
            :,
            j
        ] += qmean

        rhs_beta += qy

        mask[
            j
        ] = 1.0

        rhs_g[
            j
        ] = stats[
            node
        ][
            "mean_y"
        ]


    ridge = np.eye(
        p + 1
    )

    ridge[
        0,
        0
    ] = 0.0


    A11 += (
        lambda_beta
        * ridge
    )


    A22 = (
        np.diag(
            mask
        )

        +
        lambda_lap
        * L_kron_scaled
    )


    size = (
        p + 1
        +
        N_BOUNDARY
        +
        1
    )


    A = np.zeros(
        (
            size,
            size
        )
    )


    rhs = np.zeros(
        size
    )


    q = p + 1


    A[
        :q,
        :q
    ] = A11


    A[
        :q,
        q:q + N_BOUNDARY
    ] = A12


    A[
        q:q + N_BOUNDARY,
        :q
    ] = A12.T


    A[
        q:q + N_BOUNDARY,
        q:q + N_BOUNDARY
    ] = A22


    A[
        q:q + N_BOUNDARY,
        -1
    ] = 1.0


    A[
        -1,
        q:q + N_BOUNDARY
    ] = 1.0


    rhs[
        :q
    ] = rhs_beta


    rhs[
        q:q + N_BOUNDARY
    ] = rhs_g


    solution = np.linalg.solve(
        A,
        rhs
    )


    beta = solution[
        :q
    ]


    g = solution[
        q:q + N_BOUNDARY
    ]


    return {

        "beta":
            beta,

        "g":
            g,

        "mean_x":
            mean_x,

        "std_x":
            std_x
    }


def node_mse(
    node,
    stat,
    model,
    use_graph=False
):

    qmean, q2, qy = (
        standard_moments(

            stat,

            model[
                "mean_x"
            ],

            model[
                "std_x"
            ]
        )
    )


    coef = model[
        "beta"
    ].copy()


    if use_graph:

        coef[
            0
        ] += (
            model["g"][
                boundary_index[
                    node
                ]
            ]
        )


    mse = (

        stat[
            "mean_y2"
        ]

        - 2.0
        * coef.dot(
            qy
        )

        + coef.dot(
            q2 @ coef
        )
    )


    return max(
        float(mse),
        0.0
    )


def tune_inner_loo(
    stats,
    outer_train_nodes
):

    linear_scores = {}


    for alpha in LINEAR_ALPHA_GRID:

        fold_mse = []


        for validation_node in outer_train_nodes:

            train_nodes = [
                node
                for node
                in outer_train_nodes
                if node != validation_node
            ]


            model = fit_linear_model(
                stats,
                train_nodes,
                alpha
            )


            fold_mse.append(

                node_mse(
                    validation_node,
                    stats[
                        validation_node
                    ],
                    model,
                    use_graph=False
                )

            )


        linear_scores[
            alpha
        ] = np.mean(
            fold_mse
        )


    best_alpha = min(
        linear_scores,
        key=
            linear_scores.get
    )


    graph_scores = {}


    for lambda_lap in GRAPH_LAMBDA_GRID:

        for lambda_beta in GRAPH_BETA_GRID:

            fold_mse = []


            for validation_node in outer_train_nodes:

                train_nodes = [

                    node

                    for node
                    in outer_train_nodes

                    if node != validation_node

                ]


                model = fit_graph_model(

                    stats,

                    train_nodes,

                    lambda_lap=
                        lambda_lap,

                    lambda_beta=
                        lambda_beta
                )


                fold_mse.append(

                    node_mse(

                        validation_node,

                        stats[
                            validation_node
                        ],

                        model,

                        use_graph=True
                    )

                )


            graph_scores[
                (
                    lambda_lap,
                    lambda_beta
                )
            ] = np.mean(
                fold_mse
            )


    best_graph = min(
        graph_scores,
        key=
            graph_scores.get
    )


    return (
        best_alpha,
        best_graph
    )


def predict_rows(
    df,
    features,
    model,
    graph=False
):

    X = (
        df[
            features
        ]
        .to_numpy(
            dtype=float
        )
    )


    Z = (

        X
        -
        model[
            "mean_x"
        ]

    ) / model[
        "std_x"
    ]


    Q = np.column_stack(
        (
            np.ones(
                len(df)
            ),
            Z
        )
    )


    pred = (
        Q
        @ model[
            "beta"
        ]
    )


    if graph:

        effects = np.array(
            [
                model["g"][
                    boundary_index[
                        int(node)
                    ]
                ]

                for node
                in df["node"]
            ]
        )


        pred += effects


    return pred


def nested_loo_full_graph(
    df,
    target,
    features
):

    stats = make_node_stats(
        df,
        target,
        features
    )


    usable_nodes = sorted(
        stats.keys()
    )


    print(
        "\n",
        "=" * 70
    )

    print(
        "TARGET:",
        target
    )

    print(
        "Predictors:",
        features
    )

    print(
        "LOO nodes:",
        len(
            usable_nodes
        )
    )

    print(
        "Full graph nodes:",
        len(
            full_node_order
        )
    )


    prediction_parts = []

    tuning_rows = []


    for fold, test_node in enumerate(
        usable_nodes,
        start=1
    ):

        print(
            f"  Outer fold {fold}/{len(usable_nodes)} "
            f"- node {test_node}"
        )


        train_nodes = [

            node

            for node
            in usable_nodes

            if node != test_node

        ]


        (
            best_linear_alpha,
            best_graph
        ) = tune_inner_loo(

            stats,
            train_nodes
        )


        (
            best_lambda_lap,
            best_lambda_beta
        ) = best_graph


        linear_model = fit_linear_model(

            stats,

            train_nodes,

            best_linear_alpha
        )


        graph_model = fit_graph_model(

            stats,

            train_nodes,

            lambda_lap=
                best_lambda_lap,

            lambda_beta=
                best_lambda_beta
        )


        test_df = (
            df[
                df["node"]
                == test_node
            ]
            .copy()
        )


        linear_pred = predict_rows(

            test_df,

            features,

            linear_model,

            graph=False
        )


        graph_pred = predict_rows(

            test_df,

            features,

            graph_model,

            graph=True
        )


        fold_df = pd.DataFrame({

            "node":
                test_df[
                    "node"
                ].to_numpy(),

            "target":
                target,

            "y_true":
                test_df[
                    target
                ].to_numpy(),

            "linear_prediction":
                linear_pred,

            "graph_prediction":
                graph_pred
        })


        prediction_parts.append(
            fold_df
        )


        tuning_rows.append({

            "heldout_node":
                test_node,

            "linear_alpha":
                best_linear_alpha,

            "graph_lambda_lap":
                best_lambda_lap,

            "graph_lambda_beta":
                best_lambda_beta
        })


    predictions = pd.concat(
        prediction_parts,
        ignore_index=True
    )


    counts = (
        predictions
        .groupby(
            "node"
        )
        .size()
    )


    predictions[
        "weight"
    ] = (
        predictions[
            "node"
        ]
        .map(
            1.0
            / counts
        )
    )


    w = predictions[
        "weight"
    ].to_numpy()


    y = predictions[
        "y_true"
    ].to_numpy()


    lp = predictions[
        "linear_prediction"
    ].to_numpy()


    gp = predictions[
        "graph_prediction"
    ].to_numpy()


    linear_rmse = np.sqrt(
        mean_squared_error(
            y,
            lp,
            sample_weight=w
        )
    )


    graph_rmse = np.sqrt(
        mean_squared_error(
            y,
            gp,
            sample_weight=w
        )
    )


    linear_r2 = r2_score(
        y,
        lp,
        sample_weight=w
    )


    graph_r2 = r2_score(
        y,
        gp,
        sample_weight=w
    )


    result = {

        "Target":
            target,

        "Nodes":
            len(
                usable_nodes
            ),

        "Linear R2":
            linear_r2,

        "Graph R2":
            graph_r2,

        "Linear RMSE":
            linear_rmse,

        "Graph RMSE":
            graph_rmse
    }


    return (
        result,
        predictions,
        pd.DataFrame(
            tuning_rows
        )
    )

In [8]:
full_graph_results = []

full_graph_predictions = {}

full_graph_tuning = {}


for target, features in TARGET_FEATURES.items():

    df = target_datasets[
        target
    ]


    result, predictions, tuning = (
        nested_loo_full_graph(

            df=df,

            target=target,

            features=features
        )
    )


    full_graph_results.append(
        result
    )


    full_graph_predictions[
        target
    ] = predictions


    full_graph_tuning[
        target
    ] = tuning


full_graph_results = pd.DataFrame(
    full_graph_results
)


target_order = [
    "DO concentration",
    "Water temperature",
    "pH",
    "Specific conductance",
    "Turbidity"
]


full_graph_results[
    "Target"
] = pd.Categorical(

    full_graph_results[
        "Target"
    ],

    categories=
        target_order,

    ordered=True
)


full_graph_results = (

    full_graph_results

    .sort_values(
        "Target"
    )

    .reset_index(
        drop=True
    )
)


display(
    full_graph_results[
        [
            "Target",
            "Nodes",
            "Linear R2",
            "Graph R2",
            "Linear RMSE",
            "Graph RMSE"
        ]
    ]
    .style
    .format({

        "Linear R2":
            "{:.3f}",

        "Graph R2":
            "{:.3f}",

        "Linear RMSE":
            "{:.4f}",

        "Graph RMSE":
            "{:.4f}"
    })
)


full_graph_results.to_csv(
    "full_graph_nested_loo_results.csv",
    index=False
)


TARGET: DO concentration
Predictors: ['Water temperature', 'Specific conductance']
LOO nodes: 17
Full graph nodes: 67401
  Outer fold 1/17 - node 67739434
  Outer fold 2/17 - node 81497938
  Outer fold 3/17 - node 82982643
  Outer fold 4/17 - node 83027786
  Outer fold 5/17 - node 83222751
  Outer fold 6/17 - node 84197086
  Outer fold 7/17 - node 88786999
  Outer fold 8/17 - node 90706215
  Outer fold 9/17 - node 92116891
  Outer fold 10/17 - node 93300040
  Outer fold 11/17 - node 93392084
  Outer fold 12/17 - node 96496279
  Outer fold 13/17 - node 97532807
  Outer fold 14/17 - node 126377584
  Outer fold 15/17 - node 131520862
  Outer fold 16/17 - node 186559442
  Outer fold 17/17 - node 203177183

TARGET: Water temperature
Predictors: ['Specific conductance']
LOO nodes: 24
Full graph nodes: 67401
  Outer fold 1/24 - node 47787846
  Outer fold 2/24 - node 58738608
  Outer fold 3/24 - node 67739434
  Outer fold 4/24 - node 75465016
  Outer fold 5/24 - node 81497938
  Outer fold 6/2

,Target,Nodes,Linear R2,Graph R2,Linear RMSE,Graph RMSE
0,DO concentration,17,0.621,0.613,1.0921,1.1026
1,Water temperature,24,-0.056,-0.035,7.2828,7.2100
2,pH,19,-0.035,0.074,0.2411,0.2280
3,Specific conductance,24,-0.089,0.091,462.9602,423.0154
4,Turbidity,17,0.023,-0.049,559.4347,579.5058


In [9]:
from scipy import sparse
from scipy.sparse.linalg import splu


compressed_node_order = list(
    G_master.nodes()
)

compressed_node_index = {
    node: i
    for i, node
    in enumerate(compressed_node_order)
}

N_COMPRESSED = len(
    compressed_node_order
)


rows = []
cols = []
weights = []


G_master_und = G_master.to_undirected(
    as_view=True
)


for u, v, data in G_master_und.edges(
    data=True
):

    length_m = data.get(
        "length_m",
        None
    )


    if (
        length_m is None
        or not np.isfinite(float(length_m))
        or float(length_m) <= 0
    ):

        udata = G_master.nodes[u]
        vdata = G_master.nodes[v]

        length_m = haversine_m(
            udata["longitude"],
            udata["latitude"],
            vdata["longitude"],
            vdata["latitude"]
        )


    distance_km = max(
        float(length_m) / 1000.0,
        1e-8
    )


    w = (
        1.0
        /
        distance_km
    )


    i = compressed_node_index[u]
    j = compressed_node_index[v]


    rows.extend([
        i,
        j
    ])

    cols.extend([
        j,
        i
    ])

    weights.extend([
        w,
        w
    ])


W_compressed = sparse.coo_matrix(
    (
        weights,
        (
            rows,
            cols
        )
    ),
    shape=(
        N_COMPRESSED,
        N_COMPRESSED
    )
).tocsr()


compressed_degree = np.asarray(
    W_compressed.sum(axis=1)
).ravel()


L_compressed = (
    sparse.diags(
        compressed_degree
    )
    -
    W_compressed
).tocsr()


print(
    "Compressed Laplacian:",
    L_compressed.shape
)

print(
    "Nonzero entries:",
    L_compressed.nnz
)

Compressed Laplacian: (67, 67)
Nonzero entries: 199


In [10]:
compressed_boundary_nodes = sorted(
    all_wq_nodes
)


missing_from_compressed = (
    set(compressed_boundary_nodes)
    -
    set(compressed_node_order)
)


print(
    "Missing WQ nodes from compressed graph:",
    missing_from_compressed
)


compressed_boundary_idx = np.array(
    [
        compressed_node_index[node]
        for node
        in compressed_boundary_nodes
    ],
    dtype=int
)


all_compressed_idx = np.arange(
    N_COMPRESSED
)


compressed_interior_mask = np.ones(
    N_COMPRESSED,
    dtype=bool
)

compressed_interior_mask[
    compressed_boundary_idx
] = False


compressed_interior_idx = all_compressed_idx[
    compressed_interior_mask
]


C_BB = (
    L_compressed[
        compressed_boundary_idx,
        :
    ][
        :,
        compressed_boundary_idx
    ]
    .tocsc()
)


C_BI = (
    L_compressed[
        compressed_boundary_idx,
        :
    ][
        :,
        compressed_interior_idx
    ]
    .tocsc()
)


C_IB = (
    L_compressed[
        compressed_interior_idx,
        :
    ][
        :,
        compressed_boundary_idx
    ]
    .tocsc()
)


C_II = (
    L_compressed[
        compressed_interior_idx,
        :
    ][
        :,
        compressed_interior_idx
    ]
    .tocsc()
)


compressed_lu = splu(
    C_II
)


compressed_solution = compressed_lu.solve(
    C_IB.toarray()
)


L_kron_compressed = (
    C_BB.toarray()
    -
    C_BI @ compressed_solution
)


L_kron_compressed = np.asarray(
    L_kron_compressed
)


L_kron_compressed = (
    L_kron_compressed
    +
    L_kron_compressed.T
) / 2.0


for i in range(
    len(compressed_boundary_nodes)
):

    off_diag_sum = (
        L_kron_compressed[i, :].sum()
        -
        L_kron_compressed[i, i]
    )

    L_kron_compressed[
        i,
        i
    ] = -off_diag_sum


compressed_positive_diag = np.diag(
    L_kron_compressed
)

compressed_positive_diag = compressed_positive_diag[
    compressed_positive_diag > 1e-12
]


COMPRESSED_SCALE = (
    compressed_positive_diag.mean()
)


L_kron_compressed_scaled = (
    L_kron_compressed
    /
    COMPRESSED_SCALE
)


print(
    "Compressed effective Laplacian:",
    L_kron_compressed_scaled.shape
)

print(
    "Maximum row-sum error:",
    np.max(
        np.abs(
            L_kron_compressed_scaled.sum(
                axis=1
            )
        )
    )
)

Missing WQ nodes from compressed graph: set()
Compressed effective Laplacian: (26, 26)
Maximum row-sum error: 4.440892098500626e-16


In [11]:
difference = (
    L_kron_scaled
    -
    L_kron_compressed_scaled
)


absolute_max_difference = np.max(
    np.abs(
        difference
    )
)


frobenius_difference = np.linalg.norm(
    difference,
    ord="fro"
)


relative_frobenius_difference = (
    frobenius_difference
    /
    np.linalg.norm(
        L_kron_scaled,
        ord="fro"
    )
)


print(
    "Maximum absolute matrix difference:",
    absolute_max_difference
)

print(
    "Frobenius difference:",
    frobenius_difference
)

print(
    "Relative Frobenius difference:",
    relative_frobenius_difference
)

Maximum absolute matrix difference: 2.807531984672096e-12
Frobenius difference: 6.588950514700009e-12
Relative Frobenius difference: 5.656678447544951e-13


In [12]:
def fit_graph_model_with_L(
    stats,
    train_nodes,
    lambda_lap,
    lambda_beta,
    effective_L
):

    mean_x, std_x = (
        training_scaling(
            stats,
            train_nodes
        )
    )


    p = len(
        mean_x
    )


    A11 = np.zeros(
        (
            p + 1,
            p + 1
        )
    )


    A12 = np.zeros(
        (
            p + 1,
            N_BOUNDARY
        )
    )


    rhs_beta = np.zeros(
        p + 1
    )


    mask = np.zeros(
        N_BOUNDARY
    )


    rhs_g = np.zeros(
        N_BOUNDARY
    )


    for node in train_nodes:

        qmean, q2, qy = (
            standard_moments(
                stats[node],
                mean_x,
                std_x
            )
        )


        j = boundary_index[
            node
        ]


        A11 += q2

        A12[:, j] += qmean

        rhs_beta += qy

        mask[j] = 1.0

        rhs_g[j] = (
            stats[node][
                "mean_y"
            ]
        )


    ridge = np.eye(
        p + 1
    )

    ridge[
        0,
        0
    ] = 0.0


    A11 += (
        lambda_beta
        *
        ridge
    )


    A22 = (
        np.diag(
            mask
        )
        +
        lambda_lap
        *
        effective_L
    )


    q = p + 1


    size = (
        q
        +
        N_BOUNDARY
        +
        1
    )


    A = np.zeros(
        (
            size,
            size
        )
    )


    rhs = np.zeros(
        size
    )


    A[
        :q,
        :q
    ] = A11


    A[
        :q,
        q:q + N_BOUNDARY
    ] = A12


    A[
        q:q + N_BOUNDARY,
        :q
    ] = A12.T


    A[
        q:q + N_BOUNDARY,
        q:q + N_BOUNDARY
    ] = A22


    A[
        q:q + N_BOUNDARY,
        -1
    ] = 1.0


    A[
        -1,
        q:q + N_BOUNDARY
    ] = 1.0


    rhs[
        :q
    ] = rhs_beta


    rhs[
        q:q + N_BOUNDARY
    ] = rhs_g


    solution = np.linalg.solve(
        A,
        rhs
    )


    return {
        "beta":
            solution[:q],

        "g":
            solution[
                q:q + N_BOUNDARY
            ],

        "mean_x":
            mean_x,

        "std_x":
            std_x
    }

In [13]:
def tune_inner_loo_with_L(
    stats,
    outer_train_nodes,
    effective_L
):

    linear_scores = {}


    for alpha in LINEAR_ALPHA_GRID:

        errors = []


        for validation_node in outer_train_nodes:

            train_nodes = [
                node
                for node
                in outer_train_nodes
                if node != validation_node
            ]


            model = fit_linear_model(
                stats,
                train_nodes,
                alpha
            )


            errors.append(
                node_mse(
                    validation_node,
                    stats[
                        validation_node
                    ],
                    model,
                    use_graph=False
                )
            )


        linear_scores[
            alpha
        ] = np.mean(
            errors
        )


    best_alpha = min(
        linear_scores,
        key=linear_scores.get
    )


    graph_scores = {}


    for lambda_lap in GRAPH_LAMBDA_GRID:

        for lambda_beta in GRAPH_BETA_GRID:

            errors = []


            for validation_node in outer_train_nodes:

                train_nodes = [
                    node
                    for node
                    in outer_train_nodes
                    if node != validation_node
                ]


                model = fit_graph_model_with_L(
                    stats=stats,
                    train_nodes=train_nodes,
                    lambda_lap=lambda_lap,
                    lambda_beta=lambda_beta,
                    effective_L=effective_L
                )


                errors.append(
                    node_mse(
                        validation_node,
                        stats[
                            validation_node
                        ],
                        model,
                        use_graph=True
                    )
                )


            graph_scores[
                (
                    lambda_lap,
                    lambda_beta
                )
            ] = np.mean(
                errors
            )


    best_graph = min(
        graph_scores,
        key=graph_scores.get
    )


    return (
        best_alpha,
        best_graph
    )

# New Edge Weights

In [14]:
import rasterio
import numpy as np
import pandas as pd


MERIT_FILE = "merit_colorado_float_native.tif"


coords = [
    (
        float(G_full.nodes[node]["longitude"]),
        float(G_full.nodes[node]["latitude"])
    )
    for node in full_node_order
]


with rasterio.open(MERIT_FILE) as src:

    samples = np.asarray(
        list(
            src.sample(
                coords,
                indexes=[1, 2, 3, 4]
            )
        ),
        dtype=float
    )


upa_values = samples[:, 0]
elevation_values = samples[:, 1]
width_values = samples[:, 2]
hand_values = samples[:, 3]


for i, node in enumerate(full_node_order):

    G_full.nodes[node]["upstream_area_km2"] = upa_values[i]
    G_full.nodes[node]["elevation_m"] = elevation_values[i]
    G_full.nodes[node]["river_width_m"] = width_values[i]
    G_full.nodes[node]["hand_m"] = hand_values[i]


print("MERIT features attached.")
print()
print("UPA range:", np.nanmin(upa_values), np.nanmax(upa_values))
print("Width range:", np.nanmin(width_values), np.nanmax(width_values))
print("Elevation range:", np.nanmin(elevation_values), np.nanmax(elevation_values))


# ============================================================
# Typical discharge at WQ nodes
# ============================================================

wq_discharge = (
    mapped_hourly[
        mapped_hourly["Discharge"].notna()
    ]
    .groupby("node")["Discharge"]
    .median()
    .rename("median_discharge")
)


# ============================================================
# Try to include support gauges too if their station IDs
# happen to be in the hourly dataset
# ============================================================

support_station_to_node = {}

for node, data in G_master.nodes(data=True):

    for gauge in data.get(
        "support_gauges",
        []
    ):

        station = gauge.get(
            "station_id"
        )

        if station is not None:
            support_station_to_node[
                station
            ] = node


support_hourly = hourly[
    hourly["station_id"].isin(
        support_station_to_node.keys()
    )
].copy()


support_hourly["node"] = (
    support_hourly["station_id"]
    .map(
        support_station_to_node
    )
)


support_discharge = (
    support_hourly[
        support_hourly["Discharge"].notna()
    ]
    .groupby("node")["Discharge"]
    .median()
    .rename("median_discharge")
)


print(
    "\nSupport gauges found in hourly data:",
    support_discharge.index.nunique()
)


# Combine WQ + support gauge discharge calibration locations

discharge_calibration = pd.concat(
    [
        wq_discharge,
        support_discharge
    ]
).groupby(level=0).median()


calibration_rows = []


for node, q in discharge_calibration.items():

    if (
        node not in G_full
        or
        not np.isfinite(q)
        or
        q <= 0
    ):
        continue


    area = G_full.nodes[node].get(
        "upstream_area_km2",
        np.nan
    )


    if (
        not np.isfinite(area)
        or
        area <= 0
    ):
        continue


    calibration_rows.append({
        "node": node,
        "upstream_area_km2": area,
        "median_discharge": q
    })


discharge_fit_df = pd.DataFrame(
    calibration_rows
)


print(
    "Discharge calibration nodes:",
    len(discharge_fit_df)
)

display(discharge_fit_df)


# ============================================================
# log(Q) = intercept + slope * log(UPA)
# ============================================================

x = np.log(
    discharge_fit_df[
        "upstream_area_km2"
    ].to_numpy()
)

y = np.log(
    discharge_fit_df[
        "median_discharge"
    ].to_numpy()
)


q_slope, q_intercept = np.polyfit(
    x,
    y,
    1
)


print(
    "\nlog-discharge model:"
)

print(
    "slope =",
    q_slope
)

print(
    "intercept =",
    q_intercept
)


# Predict typical discharge at every full-network node

for node in full_node_order:

    area = max(
        float(
            G_full.nodes[node][
                "upstream_area_km2"
            ]
        ),
        1e-8
    )


    predicted_q = np.exp(
        q_intercept
        +
        q_slope
        * np.log(area)
    )


    G_full.nodes[node][
        "typical_discharge"
    ] = predicted_q

MERIT features attached.

UPA range: 10002.21484375 622761.0625
Width range: 0.44999998807907104 5436.1376953125
Elevation range: 0.0 2074.400146484375

Support gauges found in hourly data: 0
Discharge calibration nodes: 22


,node,upstream_area_km2,median_discharge
0,47787846,25163.765625,787.0
1,58738608,39059.246094,854.0
2,67739434,65933.164062,1815.0
3,75465016,10074.960938,206.0
4,81497938,11368.057617,879.0
5,83222751,15555.577148,490.0
6,84197086,18153.396484,2620.0
7,88786999,20663.750000,1320.0
8,90706215,46177.554688,2300.0
9,92116891,22887.248047,329.0



log-discharge model:
slope = 0.9269594562798037
intercept = -2.784300522556832


In [15]:
from scipy import sparse
from scipy.sparse.linalg import splu


def edge_length_m(
    G,
    u,
    v,
    data
):

    length_m = data.get(
        "length_m",
        None
    )


    if (
        length_m is not None
        and
        np.isfinite(
            float(length_m)
        )
        and
        float(length_m) > 0
    ):

        return float(
            length_m
        )


    return haversine_m(

        G.nodes[u]["longitude"],
        G.nodes[u]["latitude"],

        G.nodes[v]["longitude"],
        G.nodes[v]["latitude"]
    )


edge_feature_rows = []


for u, v, data in G_full.edges(
    data=True
):

    d_m = edge_length_m(
        G_full,
        u,
        v,
        data
    )

    d_km = max(
        d_m / 1000.0,
        1e-8
    )


    Au = max(
        float(
            G_full.nodes[u][
                "upstream_area_km2"
            ]
        ),
        1e-8
    )

    Av = max(
        float(
            G_full.nodes[v][
                "upstream_area_km2"
            ]
        ),
        1e-8
    )


    Wu = max(
        float(
            G_full.nodes[u][
                "river_width_m"
            ]
        ),
        1e-8
    )

    Wv = max(
        float(
            G_full.nodes[v][
                "river_width_m"
            ]
        ),
        1e-8
    )


    Qu = max(
        float(
            G_full.nodes[u][
                "typical_discharge"
            ]
        ),
        1e-8
    )

    Qv = max(
        float(
            G_full.nodes[v][
                "typical_discharge"
            ]
        ),
        1e-8
    )


    elev_u = float(
        G_full.nodes[u][
            "elevation_m"
        ]
    )

    elev_v = float(
        G_full.nodes[v][
            "elevation_m"
        ]
    )


    # Directed edge u -> v is downstream.
    # Positive value = downhill in the graph's flow direction.
    downstream_drop_m = max(
        elev_u - elev_v,
        0.0
    )


    slope = (
        downstream_drop_m
        /
        max(
            d_m,
            1e-8
        )
    )


    edge_feature_rows.append({

        "u": u,
        "v": v,

        "distance_km": d_km,

        "log_area": np.log(
            np.sqrt(
                Au * Av
            )
        ),

        "log_width": np.log(
            np.sqrt(
                Wu * Wv
            )
        ),

        "log_discharge": np.log(
            np.sqrt(
                Qu * Qv
            )
        ),

        "log_slope": np.log1p(
            slope
        )
    })


edge_features = pd.DataFrame(
    edge_feature_rows
)


for col in [
    "log_area",
    "log_width",
    "log_discharge",
    "log_slope"
]:

    mean = edge_features[
        col
    ].mean()

    std = edge_features[
        col
    ].std()

    if std < 1e-12:
        std = 1.0


    edge_features[
        "z_" + col
    ] = (
        edge_features[col]
        -
        mean
    ) / std


display(
    edge_features.head()
)

,u,v,distance_km,log_area,log_width,log_discharge,log_slope,z_log_area,z_log_width,z_log_discharge,z_log_slope
0,36251195,36266197,0.115347,9.214059,4.267203,5.756759,0.0,-1.381305,0.226351,-1.381305,-0.168569
1,36266197,36281199,0.115347,9.214065,4.271597,5.756764,0.0,-1.381300,0.229523,-1.381300,-0.168569
2,36281199,36296201,0.115348,9.214069,4.382020,5.756768,0.0,-1.381297,0.309230,-1.381297,-0.168569
3,36296201,36311202,0.092662,9.214080,4.457472,5.756778,0.0,-1.381289,0.363694,-1.381289,-0.168569
4,36311202,36326204,0.115349,9.214101,4.377611,5.756797,0.0,-1.381273,0.306048,-1.381273,-0.168569


In [16]:
WEIGHT_SCHEMES = {

    "distance_only": {
        "area": 0.0,
        "width": 0.0,
        "discharge": 0.0,
        "slope": 0.0
    },

    "area_width": {
        "area": 0.5,
        "width": 0.5,
        "discharge": 0.0,
        "slope": 0.0
    },

    "discharge": {
        "area": 0.0,
        "width": 0.0,
        "discharge": 1.0,
        "slope": 0.0
    },

    "area_width_discharge": {
        "area": 1/3,
        "width": 1/3,
        "discharge": 1/3,
        "slope": 0.0
    },

    "all_hydro": {
        "area": 0.25,
        "width": 0.25,
        "discharge": 0.25,
        "slope": -0.25
    }
}

In [17]:
from scipy import sparse
from scipy.sparse.linalg import splu

import numpy as np
import pandas as pd


# ============================================================
# CLEAN / STABILIZE EDGE FEATURES
# ============================================================

feature_columns = [
    "log_area",
    "log_width",
    "log_discharge",
    "log_slope"
]


for col in feature_columns:

    values = edge_features[col].to_numpy(
        dtype=float
    )

    finite_values = values[
        np.isfinite(values)
    ]

    median = np.median(
        finite_values
    )

    values = np.where(
        np.isfinite(values),
        values,
        median
    )

    edge_features[col] = values


# ============================================================
# STANDARDIZE + CLIP EXTREME Z-SCORES
#
# Clipping prevents a handful of extreme river cells from
# producing exp(score) ~ 0 or exp(score) ~ infinity.
# ============================================================

for col in feature_columns:

    mean = edge_features[col].mean()

    std = edge_features[col].std()

    if (
        not np.isfinite(std)
        or std < 1e-12
    ):
        std = 1.0


    z = (
        edge_features[col]
        -
        mean
    ) / std


    edge_features[
        "z_" + col
    ] = np.clip(
        z,
        -4.0,
        4.0
    )


print(
    edge_features[
        [
            "distance_km",
            "z_log_area",
            "z_log_width",
            "z_log_discharge",
            "z_log_slope"
        ]
    ].describe()
)

        distance_km    z_log_area   z_log_width  z_log_discharge   z_log_slope
count  67400.000000  6.740000e+04  6.740000e+04     67400.000000  67400.000000
mean       0.097055  7.016873e-16  1.079519e-16         0.000000     -0.025747
std        0.019850  1.000000e+00  1.000000e+00         1.000000      0.533305
min        0.068725 -1.384016e+00 -3.458014e+00        -1.384016     -0.168569
25%        0.076338 -9.122383e-01 -8.924466e-01        -0.912238     -0.168569
50%        0.092662 -1.509860e-01 -5.591784e-02        -0.150986     -0.168569
75%        0.118483  8.742586e-01  5.552304e-01         0.874259     -0.168569
max        0.121598  1.820164e+00  3.376712e+00         1.820164      4.000000


In [18]:
WEIGHT_SCHEMES = {

    "distance_only": {
        "area": 0.0,
        "width": 0.0,
        "discharge": 0.0,
        "slope": 0.0
    },

    "area": {
        "area": 0.5,
        "width": 0.0,
        "discharge": 0.0,
        "slope": 0.0
    },

    "width": {
        "area": 0.0,
        "width": 0.5,
        "discharge": 0.0,
        "slope": 0.0
    },

    "discharge": {
        "area": 0.0,
        "width": 0.0,
        "discharge": 0.5,
        "slope": 0.0
    },

    "slope": {
        "area": 0.0,
        "width": 0.0,
        "discharge": 0.0,
        "slope": -0.5
    },

    "area_width": {
        "area": 0.35,
        "width": 0.35,
        "discharge": 0.0,
        "slope": 0.0
    },

    "area_width_discharge": {
        "area": 0.25,
        "width": 0.25,
        "discharge": 0.25,
        "slope": 0.0
    },

    "all_hydro": {
        "area": 0.20,
        "width": 0.20,
        "discharge": 0.20,
        "slope": -0.20
    }
}

In [19]:
def build_weighted_full_L(
    edge_features,
    scheme
):

    rows = []
    cols = []
    vals = []


    for row in edge_features.itertuples():

        score = (

            scheme["area"]
            * row.z_log_area

            +

            scheme["width"]
            * row.z_log_width

            +

            scheme["discharge"]
            * row.z_log_discharge

            +

            scheme["slope"]
            * row.z_log_slope
        )


        # Prevent exp() from producing extreme conductances.
        score = np.clip(
            score,
            -5.0,
            5.0
        )


        distance_km = max(
            float(row.distance_km),
            1e-6
        )


        modifier = np.exp(
            score
        )


        w = (
            modifier
            /
            distance_km
        )


        # Safety check
        if (
            not np.isfinite(w)
            or w <= 0
        ):
            raise ValueError(
                f"Invalid edge weight on "
                f"{row.u} -> {row.v}: {w}"
            )


        i = full_node_index[
            row.u
        ]

        j = full_node_index[
            row.v
        ]


        rows.extend(
            [
                i,
                j
            ]
        )

        cols.extend(
            [
                j,
                i
            ]
        )

        vals.extend(
            [
                w,
                w
            ]
        )


    W = sparse.coo_matrix(

        (
            vals,
            (
                rows,
                cols
            )
        ),

        shape=(
            len(full_node_order),
            len(full_node_order)
        )

    ).tocsr()


    degree = np.asarray(
        W.sum(axis=1)
    ).ravel()


    L = (
        sparse.diags(
            degree
        )
        -
        W
    ).tocsr()


    return L

In [20]:
def kron_reduce_to_wq(
    L,
    scheme_name=""
):

    L_BB = (
        L[
            boundary_full_idx,
            :
        ][
            :,
            boundary_full_idx
        ]
        .tocsc()
    )


    L_BI = (
        L[
            boundary_full_idx,
            :
        ][
            :,
            interior_full_idx
        ]
        .tocsc()
    )


    L_IB = (
        L[
            interior_full_idx,
            :
        ][
            :,
            boundary_full_idx
        ]
        .tocsc()
    )


    L_II = (
        L[
            interior_full_idx,
            :
        ][
            :,
            interior_full_idx
        ]
        .tocsc()
    )


    print(
        f"  L_II shape: {L_II.shape}"
    )

    print(
        f"  L_II nnz: {L_II.nnz}"
    )

    diag = L_II.diagonal()


    print(
        "  diagonal range:",
        float(np.min(diag)),
        "to",
        float(np.max(diag))
    )


    print(
        "  zero diagonal entries:",
        int(
            np.sum(
                diag <= 0
            )
        )
    )


    if not np.isfinite(
        L_II.data
    ).all():

        raise ValueError(
            f"{scheme_name}: L_II contains "
            "NaN or infinity."
        )


    if np.any(
        diag <= 0
    ):

        raise ValueError(
            f"{scheme_name}: L_II contains "
            "non-positive diagonal entries."
        )


    try:

        lu = splu(
            L_II
        )

    except RuntimeError as e:

        raise RuntimeError(
            f"Kron reduction failed for "
            f"scheme '{scheme_name}'."
        ) from e


    solution = lu.solve(
        L_IB.toarray()
    )


    L_eff = (
        L_BB.toarray()
        -
        L_BI @ solution
    )


    L_eff = np.asarray(
        L_eff
    )


    L_eff = (
        L_eff
        +
        L_eff.T
    ) / 2.0


    # Numerical cleanup:
    # enforce Laplacian row sums exactly.
    for i in range(
        L_eff.shape[0]
    ):

        off_diag_sum = (
            L_eff[i, :].sum()
            -
            L_eff[i, i]
        )

        L_eff[
            i,
            i
        ] = -off_diag_sum


    diag_eff = np.diag(
        L_eff
    )


    valid_diag = diag_eff[
        diag_eff > 1e-12
    ]


    if len(valid_diag) == 0:

        raise ValueError(
            f"{scheme_name}: reduced "
            "Laplacian has no positive diagonal."
        )


    scale = np.mean(
        valid_diag
    )


    L_eff_scaled = (
        L_eff
        /
        scale
    )


    return L_eff_scaled

In [21]:
effective_laplacians = {}


for name, scheme in WEIGHT_SCHEMES.items():

    print(
        "\n"
        + "=" * 60
    )

    print(
        "Building:",
        name
    )

    print(
        "=" * 60
    )


    L_scheme = build_weighted_full_L(
        edge_features,
        scheme
    )


    print(
        "Full L finite:",
        np.isfinite(
            L_scheme.data
        ).all()
    )

    print(
        "Full L nnz:",
        L_scheme.nnz
    )


    L_eff = kron_reduce_to_wq(
        L_scheme,
        scheme_name=name
    )


    effective_laplacians[
        name
    ] = L_eff


    eigvals = np.linalg.eigvalsh(
        L_eff
    )


    print(
        "Smallest eigenvalues:"
    )

    print(
        eigvals[:5]
    )


    print(
        "Finished:",
        name
    )


print(
    "\nSuccessfully created:"
)

print(
    list(
        effective_laplacians.keys()
    )
)


Building: distance_only
Full L finite: True
Full L nnz: 202201
  L_II shape: (67375, 67375)
  L_II nnz: 202071
  diagonal range: 8.420425105308496 to 36.68913097661883
  zero diagonal entries: 0
Smallest eigenvalues:
[-2.75278931e-16  5.43830838e-03  8.04522277e-03  1.10802187e-02
  4.11998132e-02]
Finished: distance_only

Building: area
Full L finite: True
Full L nnz: 202201
  L_II shape: (67375, 67375)
  L_II nnz: 202071
  diagonal range: 4.2220452565305004 to 77.10203314124
  zero diagonal entries: 0
Smallest eigenvalues:
[-5.84322571e-17  1.03542690e-02  1.53157948e-02  2.31768657e-02
  5.16242721e-02]
Finished: area

Building: width
Full L finite: True
Full L nnz: 202201
  L_II shape: (67375, 67375)
  L_II nnz: 202071
  diagonal range: 1.9241836009166058 to 125.4836706452341
  zero diagonal entries: 0
Smallest eigenvalues:
[-6.65068222e-16  8.26244410e-03  1.14030652e-02  1.45354118e-02
  4.90997047e-02]
Finished: width

Building: discharge
Full L finite: True
Full L nnz: 202201


In [22]:
effective_laplacians_clean = {
    "distance_only":
        effective_laplacians["distance_only"],

    "area":
        effective_laplacians["area"],

    "width":
        effective_laplacians["width"],

    "slope":
        effective_laplacians["slope"],

    "area_width":
        effective_laplacians["area_width"]
}

In [23]:
def tune_inner_loo_weighted(
    stats,
    outer_train_nodes
):

    linear_scores = {}

    for alpha in LINEAR_ALPHA_GRID:

        errors = []

        for validation_node in outer_train_nodes:

            train_nodes = [
                node
                for node in outer_train_nodes
                if node != validation_node
            ]

            model = fit_linear_model(
                stats,
                train_nodes,
                alpha
            )

            errors.append(
                node_mse(
                    validation_node,
                    stats[validation_node],
                    model,
                    use_graph=False
                )
            )

        linear_scores[alpha] = np.mean(errors)


    best_alpha = min(
        linear_scores,
        key=linear_scores.get
    )


    graph_scores = {}

    for scheme_name, effective_L in effective_laplacians_clean.items():

        for lambda_lap in GRAPH_LAMBDA_GRID:

            for lambda_beta in GRAPH_BETA_GRID:

                errors = []

                for validation_node in outer_train_nodes:

                    train_nodes = [
                        node
                        for node in outer_train_nodes
                        if node != validation_node
                    ]

                    model = fit_graph_model_with_L(
                        stats=stats,
                        train_nodes=train_nodes,
                        lambda_lap=lambda_lap,
                        lambda_beta=lambda_beta,
                        effective_L=effective_L
                    )

                    errors.append(
                        node_mse(
                            validation_node,
                            stats[validation_node],
                            model,
                            use_graph=True
                        )
                    )

                graph_scores[
                    (
                        scheme_name,
                        lambda_lap,
                        lambda_beta
                    )
                ] = np.mean(errors)


    best_graph = min(
        graph_scores,
        key=graph_scores.get
    )

    return (
        best_alpha,
        best_graph
    )

In [24]:
def nested_loo_hydrologic(
    df,
    target,
    features
):

    stats = make_node_stats(
        df,
        target,
        features
    )

    usable_nodes = sorted(
        stats.keys()
    )

    print("\n" + "=" * 70)
    print("TARGET:", target)
    print("Predictors:", features)
    print("LOO nodes:", len(usable_nodes))
    print("=" * 70)


    prediction_parts = []
    tuning_rows = []


    for fold, test_node in enumerate(
        usable_nodes,
        start=1
    ):

        print(
            f"  Outer fold {fold}/{len(usable_nodes)} "
            f"- node {test_node}"
        )


        train_nodes = [
            node
            for node in usable_nodes
            if node != test_node
        ]


        (
            best_linear_alpha,
            best_graph
        ) = tune_inner_loo_weighted(
            stats,
            train_nodes
        )


        (
            best_scheme,
            best_lambda_lap,
            best_lambda_beta
        ) = best_graph


        linear_model = fit_linear_model(
            stats,
            train_nodes,
            best_linear_alpha
        )


        graph_model = fit_graph_model_with_L(
            stats=stats,
            train_nodes=train_nodes,
            lambda_lap=best_lambda_lap,
            lambda_beta=best_lambda_beta,
            effective_L=
                effective_laplacians_clean[
                    best_scheme
                ]
        )


        test_df = (
            df[
                df["node"] == test_node
            ]
            .copy()
        )


        linear_pred = predict_rows(
            test_df,
            features,
            linear_model,
            graph=False
        )


        graph_pred = predict_rows(
            test_df,
            features,
            graph_model,
            graph=True
        )


        fold_df = pd.DataFrame({

            "node":
                test_df["node"].to_numpy(),

            "target":
                target,

            "y_true":
                test_df[target].to_numpy(),

            "linear_prediction":
                linear_pred,

            "graph_prediction":
                graph_pred,

            "selected_scheme":
                best_scheme
        })


        prediction_parts.append(
            fold_df
        )


        tuning_rows.append({

            "heldout_node":
                test_node,

            "linear_alpha":
                best_linear_alpha,

            "graph_scheme":
                best_scheme,

            "graph_lambda_lap":
                best_lambda_lap,

            "graph_lambda_beta":
                best_lambda_beta
        })


    predictions = pd.concat(
        prediction_parts,
        ignore_index=True
    )


    tuning = pd.DataFrame(
        tuning_rows
    )


    counts = (
        predictions
        .groupby("node")
        .size()
    )


    predictions["weight"] = (
        predictions["node"]
        .map(
            1.0 / counts
        )
    )


    w = predictions[
        "weight"
    ].to_numpy()


    y = predictions[
        "y_true"
    ].to_numpy()


    lp = predictions[
        "linear_prediction"
    ].to_numpy()


    gp = predictions[
        "graph_prediction"
    ].to_numpy()


    linear_rmse = np.sqrt(
        mean_squared_error(
            y,
            lp,
            sample_weight=w
        )
    )


    graph_rmse = np.sqrt(
        mean_squared_error(
            y,
            gp,
            sample_weight=w
        )
    )


    linear_r2 = r2_score(
        y,
        lp,
        sample_weight=w
    )


    graph_r2 = r2_score(
        y,
        gp,
        sample_weight=w
    )


    result = {

        "Target":
            target,

        "Nodes":
            len(usable_nodes),

        "Linear R2":
            linear_r2,

        "Graph R2":
            graph_r2,

        "Linear RMSE":
            linear_rmse,

        "Graph RMSE":
            graph_rmse,

        "Graph RMSE Improvement %":
            (
                (
                    linear_rmse
                    - graph_rmse
                )
                /
                linear_rmse
                * 100
            )
    }


    return (
        result,
        predictions,
        tuning
    )

In [25]:
hydro_results = []

hydro_predictions = {}

hydro_tuning = {}


for target, features in TARGET_FEATURES.items():

    result, predictions, tuning = (
        nested_loo_hydrologic(
            df=target_datasets[target],
            target=target,
            features=features
        )
    )


    hydro_results.append(
        result
    )

    hydro_predictions[target] = (
        predictions
    )

    hydro_tuning[target] = (
        tuning
    )


hydro_results = pd.DataFrame(
    hydro_results
)


target_order = [
    "DO concentration",
    "Water temperature",
    "pH",
    "Specific conductance",
    "Turbidity"
]


hydro_results["Target"] = pd.Categorical(
    hydro_results["Target"],
    categories=target_order,
    ordered=True
)


hydro_results = (
    hydro_results
    .sort_values("Target")
    .reset_index(drop=True)
)


display(
    hydro_results.style.format({

        "Linear R2":
            "{:.3f}",

        "Graph R2":
            "{:.3f}",

        "Linear RMSE":
            "{:.4f}",

        "Graph RMSE":
            "{:.4f}",

        "Graph RMSE Improvement %":
            "{:.2f}%"
    })
)


TARGET: DO concentration
Predictors: ['Water temperature', 'Specific conductance']
LOO nodes: 17
  Outer fold 1/17 - node 67739434
  Outer fold 2/17 - node 81497938
  Outer fold 3/17 - node 82982643
  Outer fold 4/17 - node 83027786
  Outer fold 5/17 - node 83222751
  Outer fold 6/17 - node 84197086
  Outer fold 7/17 - node 88786999
  Outer fold 8/17 - node 90706215
  Outer fold 9/17 - node 92116891
  Outer fold 10/17 - node 93300040
  Outer fold 11/17 - node 93392084
  Outer fold 12/17 - node 96496279
  Outer fold 13/17 - node 97532807
  Outer fold 14/17 - node 126377584
  Outer fold 15/17 - node 131520862
  Outer fold 16/17 - node 186559442
  Outer fold 17/17 - node 203177183

TARGET: Water temperature
Predictors: ['Specific conductance']
LOO nodes: 24
  Outer fold 1/24 - node 47787846
  Outer fold 2/24 - node 58738608
  Outer fold 3/24 - node 67739434
  Outer fold 4/24 - node 75465016
  Outer fold 5/24 - node 81497938
  Outer fold 6/24 - node 82982643
  Outer fold 7/24 - node 83027

,Target,Nodes,Linear R2,Graph R2,Linear RMSE,Graph RMSE,Graph RMSE Improvement %
0,DO concentration,17,0.621,0.607,1.0921,1.1125,-1.87%
1,Water temperature,24,-0.056,-0.058,7.2828,7.2894,-0.09%
2,pH,19,-0.035,0.074,0.2411,0.2280,5.43%
3,Specific conductance,24,-0.089,0.106,462.9602,419.4028,9.41%
4,Turbidity,17,0.023,-0.036,559.4347,576.0086,-2.96%


In [26]:
scheme_rows = []


for target, tuning in hydro_tuning.items():

    counts = (
        tuning[
            "graph_scheme"
        ]
        .value_counts()
    )


    for scheme, count in counts.items():

        scheme_rows.append({

            "Target":
                target,

            "Weighting":
                scheme,

            "Selected folds":
                count,

            "Total folds":
                len(tuning),

            "Selection %":
                (
                    count
                    /
                    len(tuning)
                    * 100
                )
        })


scheme_selection = (
    pd.DataFrame(
        scheme_rows
    )
    .sort_values(
        [
            "Target",
            "Selected folds"
        ],
        ascending=[
            True,
            False
        ]
    )
)


display(
    scheme_selection.style.format({
        "Selection %":
            "{:.1f}%"
    })
)

,Target,Weighting,Selected folds,Total folds,Selection %
0,DO concentration,width,13,17,76.5%
1,DO concentration,area_width,3,17,17.6%
2,DO concentration,distance_only,1,17,5.9%
6,Specific conductance,area_width,22,24,91.7%
7,Specific conductance,area,2,24,8.3%
8,Turbidity,width,13,17,76.5%
9,Turbidity,area_width,2,17,11.8%
10,Turbidity,area,2,17,11.8%
3,Water temperature,area,15,24,62.5%
4,Water temperature,slope,5,24,20.8%


In [27]:
node_comparison_rows = []


for target, predictions in hydro_predictions.items():

    for node, group in predictions.groupby("node"):

        y = group["y_true"].to_numpy()

        linear_pred = (
            group["linear_prediction"]
            .to_numpy()
        )

        graph_pred = (
            group["graph_prediction"]
            .to_numpy()
        )


        linear_rmse = np.sqrt(
            np.mean(
                (y - linear_pred) ** 2
            )
        )

        graph_rmse = np.sqrt(
            np.mean(
                (y - graph_pred) ** 2
            )
        )


        improvement_pct = (
            (
                linear_rmse
                - graph_rmse
            )
            /
            linear_rmse
            * 100
        )


        node_comparison_rows.append({

            "Target":
                target,

            "Node":
                node,

            "Linear RMSE":
                linear_rmse,

            "Graph RMSE":
                graph_rmse,

            "Graph Improvement %":
                improvement_pct,

            "Graph Better":
                graph_rmse < linear_rmse
        })


node_comparison = pd.DataFrame(
    node_comparison_rows
)


summary_rows = []


for target, group in node_comparison.groupby(
    "Target"
):

    summary_rows.append({

        "Target":
            target,

        "Nodes":
            len(group),

        "Graph Better Nodes":
            int(
                group[
                    "Graph Better"
                ].sum()
            ),

        "Graph Worse Nodes":
            int(
                (
                    ~group[
                        "Graph Better"
                    ]
                ).sum()
            ),

        "Median Improvement %":
            group[
                "Graph Improvement %"
            ].median(),

        "Mean Improvement %":
            group[
                "Graph Improvement %"
            ].mean()
    })


node_summary = pd.DataFrame(
    summary_rows
)


display(
    node_summary.style.format({

        "Median Improvement %":
            "{:.2f}%",

        "Mean Improvement %":
            "{:.2f}%"
    })
)

,Target,Nodes,Graph Better Nodes,Graph Worse Nodes,Median Improvement %,Mean Improvement %
0,DO concentration,17,12,5,4.53%,-0.88%
1,Specific conductance,24,15,9,12.22%,1.17%
2,Turbidity,17,12,5,3.18%,6.11%
3,Water temperature,24,16,8,0.38%,0.21%
4,pH,19,13,6,5.86%,4.62%


In [28]:
for target in [
    "pH",
    "Specific conductance"
]:

    print(
        "\n",
        "=" * 70
    )

    print(target)

    print(
        "=" * 70
    )


    display(
        node_comparison[
            node_comparison[
                "Target"
            ] == target
        ]
        .sort_values(
            "Graph Improvement %",
            ascending=False
        )
        .style
        .format({

            "Linear RMSE":
                "{:.4f}",

            "Graph RMSE":
                "{:.4f}",

            "Graph Improvement %":
                "{:.2f}%"
        })
    )


pH


,Target,Node,Linear RMSE,Graph RMSE,Graph Improvement %,Graph Better
82,pH,67739434,0.1561,0.1019,34.73%,True
91,pH,93392084,0.2159,0.1623,24.83%,True
84,pH,82982643,0.3596,0.2884,19.82%,True
94,pH,126377584,0.2162,0.1915,11.39%,True
96,pH,131520862,0.3203,0.2870,10.39%,True
90,pH,93300040,0.2619,0.2373,9.40%,True
93,pH,97532807,0.2002,0.1836,8.29%,True
88,pH,88786999,0.2074,0.1927,7.07%,True
89,pH,90706215,0.2434,0.2284,6.19%,True
87,pH,84197086,0.2554,0.2404,5.86%,True



Specific conductance


,Target,Node,Linear RMSE,Graph RMSE,Graph Improvement %,Graph Better
42,Specific conductance,58738608,244.2415,70.4201,71.17%,True
41,Specific conductance,47787846,396.6601,150.5557,62.04%,True
51,Specific conductance,90706215,414.2697,158.0218,61.86%,True
62,Specific conductance,134075067,478.3868,186.3223,61.05%,True
61,Specific conductance,133114453,381.3972,172.5497,54.76%,True
43,Specific conductance,67739434,322.2788,165.7114,48.58%,True
59,Specific conductance,126377584,304.0550,181.7808,40.21%,True
50,Specific conductance,88786999,375.5016,255.0647,32.07%,True
52,Specific conductance,92116891,736.6181,581.8009,21.02%,True
56,Specific conductance,96751415,1559.8101,1270.8238,18.53%,True


In [29]:
import numpy as np
import pandas as pd

from itertools import product

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)


TARGET_CONFIG = {

    "DO concentration": [
        "Water temperature",
        "Specific conductance"
    ],

    "Water temperature": [
        "Specific conductance"
    ],

    "pH": [
        "Specific conductance"
    ],

    "Specific conductance": [
        "Water temperature"
    ],

    "Turbidity": [
        "Specific conductance",
        "pH"
    ]
}


model_source = node_hourly_all.copy()


model_source["time"] = pd.to_datetime(
    model_source["time"],
    utc=True
)


model_source["hour"] = (
    model_source["time"].dt.hour
    +
    model_source["time"].dt.minute / 60
)


model_source["doy"] = (
    model_source["time"]
    .dt.dayofyear
)


model_source["hour_sin"] = np.sin(
    2 * np.pi
    * model_source["hour"]
    / 24
)


model_source["hour_cos"] = np.cos(
    2 * np.pi
    * model_source["hour"]
    / 24
)


model_source["doy_sin"] = np.sin(
    2 * np.pi
    * model_source["doy"]
    / 365.25
)


model_source["doy_cos"] = np.cos(
    2 * np.pi
    * model_source["doy"]
    / 365.25
)


TIME_FEATURES = [
    "hour_sin",
    "hour_cos",
    "doy_sin",
    "doy_cos"
]


comparison_datasets = {}


for target, features in TARGET_CONFIG.items():

    df = (
        model_source[
            [
                "node",
                "time",
                target
            ]
            + features
            + TIME_FEATURES
        ]
        .dropna(
            subset=[
                target
            ] + features
        )
        .copy()
    )


    comparison_datasets[
        target
    ] = df


    print(
        target,
        "->",
        df["node"].nunique(),
        "nodes,",
        len(df),
        "node-hours"
    )

DO concentration -> 17 nodes, 114607 node-hours
Water temperature -> 24 nodes, 144980 node-hours
pH -> 19 nodes, 121011 node-hours
Specific conductance -> 24 nodes, 144980 node-hours
Turbidity -> 17 nodes, 88095 node-hours


In [30]:
def node_weights(df):

    counts = (
        df.groupby("node")
        .size()
    )


    return (
        df["node"]
        .map(
            1.0 / counts
        )
        .to_numpy(
            dtype=float
        )
    )


def weighted_scaler(
    X,
    weights
):

    weights = np.asarray(
        weights,
        dtype=float
    )


    weights = (
        weights
        /
        weights.sum()
    )


    mean = np.sum(
        X
        *
        weights[:, None],
        axis=0
    )


    variance = np.sum(
        weights[:, None]
        *
        (X - mean) ** 2,
        axis=0
    )


    std = np.sqrt(
        np.maximum(
            variance,
            1e-12
        )
    )


    return (
        mean,
        std
    )


def node_validation_mse(
    y,
    pred
):

    return np.mean(
        (
            y - pred
        ) ** 2
    )


def predict_linear_chemistry(
    train_df,
    test_df,
    target,
    features
):

    X_train = (
        train_df[
            features
        ]
        .to_numpy(
            dtype=float
        )
    )


    y_train = (
        train_df[
            target
        ]
        .to_numpy(
            dtype=float
        )
    )


    X_test = (
        test_df[
            features
        ]
        .to_numpy(
            dtype=float
        )
    )


    weights = node_weights(
        train_df
    )


    model = LinearRegression()


    model.fit(
        X_train,
        y_train,
        sample_weight=weights
    )


    return model.predict(
        X_test
    )

In [31]:
SEASONAL_ALPHA_GRID = [
    0.001,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0
]


QUADRATIC_ALPHA_GRID = [
    0.001,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0,
    30.0,
    100.0
]


GB_GRID = [

    {
        "learning_rate": lr,
        "max_leaf_nodes": leaves,
        "l2_regularization": l2
    }

    for lr, leaves, l2 in product(

        [
            0.03,
            0.05,
            0.1
        ],

        [
            7,
            15
        ],

        [
            0.1,
            1.0,
            10.0
        ]
    )
]


def predict_seasonal(
    train_df,
    test_df,
    target,
    features,
    alpha
):

    seasonal_features = (
        features
        +
        TIME_FEATURES
    )


    X_train = (
        train_df[
            seasonal_features
        ]
        .to_numpy(
            dtype=float
        )
    )


    X_test = (
        test_df[
            seasonal_features
        ]
        .to_numpy(
            dtype=float
        )
    )


    y_train = (
        train_df[
            target
        ]
        .to_numpy(
            dtype=float
        )
    )


    weights = node_weights(
        train_df
    )


    mean, std = weighted_scaler(
        X_train,
        weights
    )


    X_train = (
        X_train - mean
    ) / std


    X_test = (
        X_test - mean
    ) / std


    model = Ridge(
        alpha=alpha
    )


    model.fit(
        X_train,
        y_train,
        sample_weight=weights
    )


    return model.predict(
        X_test
    )


def predict_quadratic(
    train_df,
    test_df,
    target,
    features,
    alpha
):

    X_train = (
        train_df[
            features
        ]
        .to_numpy(
            dtype=float
        )
    )


    X_test = (
        test_df[
            features
        ]
        .to_numpy(
            dtype=float
        )
    )


    y_train = (
        train_df[
            target
        ]
        .to_numpy(
            dtype=float
        )
    )


    weights = node_weights(
        train_df
    )


    mean, std = weighted_scaler(
        X_train,
        weights
    )


    X_train = (
        X_train - mean
    ) / std


    X_test = (
        X_test - mean
    ) / std


    poly = PolynomialFeatures(
        degree=2,
        include_bias=False
    )


    X_train = poly.fit_transform(
        X_train
    )


    X_test = poly.transform(
        X_test
    )


    poly_mean, poly_std = weighted_scaler(
        X_train,
        weights
    )


    X_train = (
        X_train - poly_mean
    ) / poly_std


    X_test = (
        X_test - poly_mean
    ) / poly_std


    model = Ridge(
        alpha=alpha
    )


    model.fit(
        X_train,
        y_train,
        sample_weight=weights
    )


    return model.predict(
        X_test
    )


def predict_gradient_boosting(
    train_df,
    test_df,
    target,
    features,
    params
):

    X_train = (
        train_df[
            features
        ]
        .to_numpy(
            dtype=float
        )
    )


    X_test = (
        test_df[
            features
        ]
        .to_numpy(
            dtype=float
        )
    )


    y_train = (
        train_df[
            target
        ]
        .to_numpy(
            dtype=float
        )
    )


    weights = node_weights(
        train_df
    )


    model = HistGradientBoostingRegressor(

        learning_rate=
            params[
                "learning_rate"
            ],

        max_leaf_nodes=
            params[
                "max_leaf_nodes"
            ],

        l2_regularization=
            params[
                "l2_regularization"
            ],

        max_iter=150,

        min_samples_leaf=100,

        early_stopping=False,

        random_state=42
    )


    model.fit(
        X_train,
        y_train,
        sample_weight=weights
    )


    return model.predict(
        X_test
    )

In [32]:
def tune_standard_models(
    outer_train_df,
    target,
    features
):

    nodes = sorted(
        outer_train_df[
            "node"
        ].unique()
    )


    seasonal_scores = {}


    for alpha in SEASONAL_ALPHA_GRID:

        errors = []


        for validation_node in nodes:

            train = (
                outer_train_df[
                    outer_train_df["node"]
                    != validation_node
                ]
            )


            validation = (
                outer_train_df[
                    outer_train_df["node"]
                    == validation_node
                ]
            )


            pred = predict_seasonal(
                train,
                validation,
                target,
                features,
                alpha
            )


            errors.append(
                node_validation_mse(

                    validation[
                        target
                    ].to_numpy(),

                    pred
                )
            )


        seasonal_scores[
            alpha
        ] = np.mean(
            errors
        )


    best_seasonal = min(
        seasonal_scores,
        key=seasonal_scores.get
    )


    quadratic_scores = {}


    for alpha in QUADRATIC_ALPHA_GRID:

        errors = []


        for validation_node in nodes:

            train = (
                outer_train_df[
                    outer_train_df["node"]
                    != validation_node
                ]
            )


            validation = (
                outer_train_df[
                    outer_train_df["node"]
                    == validation_node
                ]
            )


            pred = predict_quadratic(
                train,
                validation,
                target,
                features,
                alpha
            )


            errors.append(
                node_validation_mse(

                    validation[
                        target
                    ].to_numpy(),

                    pred
                )
            )


        quadratic_scores[
            alpha
        ] = np.mean(
            errors
        )


    best_quadratic = min(
        quadratic_scores,
        key=quadratic_scores.get
    )


    gb_scores = {}


    for i, params in enumerate(
        GB_GRID
    ):

        errors = []


        for validation_node in nodes:

            train = (
                outer_train_df[
                    outer_train_df["node"]
                    != validation_node
                ]
            )


            validation = (
                outer_train_df[
                    outer_train_df["node"]
                    == validation_node
                ]
            )


            pred = predict_gradient_boosting(
                train,
                validation,
                target,
                features,
                params
            )


            errors.append(
                node_validation_mse(

                    validation[
                        target
                    ].to_numpy(),

                    pred
                )
            )


        gb_scores[
            i
        ] = np.mean(
            errors
        )


    best_gb_index = min(
        gb_scores,
        key=gb_scores.get
    )


    return {

        "seasonal_alpha":
            best_seasonal,

        "quadratic_alpha":
            best_quadratic,

        "gb_params":
            GB_GRID[
                best_gb_index
            ]
    }

In [34]:
def tune_graph_model(
    stats,
    outer_train_nodes
):

    scores = {}


    for (
        scheme_name,
        effective_L
    ) in effective_laplacians_clean.items():

        for lambda_lap in GRAPH_LAMBDA_GRID:

            for lambda_beta in GRAPH_BETA_GRID:

                errors = []


                for validation_node in outer_train_nodes:

                    inner_train_nodes = [

                        node

                        for node
                        in outer_train_nodes

                        if node
                        != validation_node
                    ]


                    model = fit_graph_model_with_L(

                        stats=stats,

                        train_nodes=
                            inner_train_nodes,

                        lambda_lap=
                            lambda_lap,

                        lambda_beta=
                            lambda_beta,

                        effective_L=
                            effective_L
                    )


                    errors.append(

                        node_mse(

                            validation_node,

                            stats[
                                validation_node
                            ],

                            model,

                            use_graph=True
                        )

                    )


                scores[
                    (
                        scheme_name,
                        lambda_lap,
                        lambda_beta
                    )
                ] = np.mean(
                    errors
                )


    return min(
        scores,
        key=scores.get
    )

In [35]:
all_predictions = []

all_tuning = []


for target, features in TARGET_CONFIG.items():

    df = comparison_datasets[
        target
    ]


    nodes = sorted(
        df[
            "node"
        ].unique()
    )


    stats = make_node_stats(
        df,
        target,
        features
    )


    print(
        "\n"
        + "=" * 75
    )

    print(
        "TARGET:",
        target
    )

    print(
        "Predictors:",
        features
    )

    print(
        "Outer LOO folds:",
        len(nodes)
    )

    print(
        "=" * 75
    )


    for fold, test_node in enumerate(
        nodes,
        start=1
    ):

        print(
            f"  Fold {fold}/{len(nodes)} "
            f"- held out node {test_node}"
        )


        train_df = (
            df[
                df["node"]
                != test_node
            ]
            .copy()
        )


        test_df = (
            df[
                df["node"]
                == test_node
            ]
            .copy()
        )


        train_nodes = [

            node

            for node in nodes

            if node != test_node

        ]


        tuned = tune_standard_models(
            train_df,
            target,
            features
        )


        (
            graph_scheme,
            graph_lambda,
            graph_beta
        ) = tune_graph_model(
            stats,
            train_nodes
        )


        linear_pred = (
            predict_linear_chemistry(

                train_df,
                test_df,
                target,
                features
            )
        )


        seasonal_pred = (
            predict_seasonal(

                train_df,
                test_df,
                target,
                features,

                tuned[
                    "seasonal_alpha"
                ]
            )
        )


        quadratic_pred = (
            predict_quadratic(

                train_df,
                test_df,
                target,
                features,

                tuned[
                    "quadratic_alpha"
                ]
            )
        )


        gb_pred = (
            predict_gradient_boosting(

                train_df,
                test_df,
                target,
                features,

                tuned[
                    "gb_params"
                ]
            )
        )


        graph_model = (
            fit_graph_model_with_L(

                stats=stats,

                train_nodes=
                    train_nodes,

                lambda_lap=
                    graph_lambda,

                lambda_beta=
                    graph_beta,

                effective_L=
                    effective_laplacians_clean[
                        graph_scheme
                    ]
            )
        )


        graph_pred = predict_rows(

            test_df,

            features,

            graph_model,

            graph=True
        )


        fold_predictions = pd.DataFrame({

            "Target":
                target,

            "node":
                test_df[
                    "node"
                ].to_numpy(),

            "time":
                test_df[
                    "time"
                ].to_numpy(),

            "y_true":
                test_df[
                    target
                ].to_numpy(),

            "Linear chemistry":
                linear_pred,

            "Linear + Laplacian":
                graph_pred,

            "Seasonal linear":
                seasonal_pred,

            "Gradient boosting":
                gb_pred,

            "Quadratic ridge":
                quadratic_pred
        })


        all_predictions.append(
            fold_predictions
        )


        all_tuning.append({

            "Target":
                target,

            "heldout_node":
                test_node,

            "seasonal_alpha":
                tuned[
                    "seasonal_alpha"
                ],

            "quadratic_alpha":
                tuned[
                    "quadratic_alpha"
                ],

            "gb_learning_rate":
                tuned[
                    "gb_params"
                ][
                    "learning_rate"
                ],

            "gb_leaf_nodes":
                tuned[
                    "gb_params"
                ][
                    "max_leaf_nodes"
                ],

            "gb_l2":
                tuned[
                    "gb_params"
                ][
                    "l2_regularization"
                ],

            "graph_scheme":
                graph_scheme,

            "graph_lambda":
                graph_lambda,

            "graph_beta":
                graph_beta
        })


model_loo_predictions = pd.concat(
    all_predictions,
    ignore_index=True
)


model_loo_tuning = pd.DataFrame(
    all_tuning
)


TARGET: DO concentration
Predictors: ['Water temperature', 'Specific conductance']
Outer LOO folds: 17
  Fold 1/17 - held out node 67739434
  Fold 2/17 - held out node 81497938
  Fold 3/17 - held out node 82982643
  Fold 4/17 - held out node 83027786
  Fold 5/17 - held out node 83222751
  Fold 6/17 - held out node 84197086
  Fold 7/17 - held out node 88786999
  Fold 8/17 - held out node 90706215
  Fold 9/17 - held out node 92116891
  Fold 10/17 - held out node 93300040
  Fold 11/17 - held out node 93392084
  Fold 12/17 - held out node 96496279
  Fold 13/17 - held out node 97532807
  Fold 14/17 - held out node 126377584
  Fold 15/17 - held out node 131520862
  Fold 16/17 - held out node 186559442
  Fold 17/17 - held out node 203177183

TARGET: Water temperature
Predictors: ['Specific conductance']
Outer LOO folds: 24
  Fold 1/24 - held out node 47787846
  Fold 2/24 - held out node 58738608
  Fold 3/24 - held out node 67739434
  Fold 4/24 - held out node 75465016
  Fold 5/24 - held out 

In [36]:
MODEL_NAMES = [
    "Linear chemistry",
    "Linear + Laplacian",
    "Seasonal linear",
    "Gradient boosting",
    "Quadratic ridge"
]


result_rows = []


for target in TARGET_CONFIG:

    target_predictions = (
        model_loo_predictions[
            model_loo_predictions[
                "Target"
            ] == target
        ]
        .copy()
    )


    counts = (
        target_predictions
        .groupby("node")
        .size()
    )


    weights = (
        target_predictions[
            "node"
        ]
        .map(
            1.0 / counts
        )
        .to_numpy()
    )


    y = (
        target_predictions[
            "y_true"
        ]
        .to_numpy()
    )


    n_nodes = (
        target_predictions[
            "node"
        ]
        .nunique()
    )


    for model_name in MODEL_NAMES:

        pred = (
            target_predictions[
                model_name
            ]
            .to_numpy()
        )


        result_rows.append({

            "Target":
                target,

            "Nodes":
                n_nodes,

            "Model":
                model_name,

            "R2":
                r2_score(
                    y,
                    pred,
                    sample_weight=weights
                ),

            "RMSE":
                np.sqrt(
                    mean_squared_error(
                        y,
                        pred,
                        sample_weight=weights
                    )
                ),

            "MAE":
                mean_absolute_error(
                    y,
                    pred,
                    sample_weight=weights
                )
        })


model_loo_results = pd.DataFrame(
    result_rows
)


display(
    model_loo_results
    .style
    .format({

        "R2":
            "{:.3f}",

        "RMSE":
            "{:.4f}",

        "MAE":
            "{:.4f}"
    })
)

,Target,Nodes,Model,R2,RMSE,MAE
0,DO concentration,17,Linear chemistry,0.621,1.0919,0.8206
1,DO concentration,17,Linear + Laplacian,0.607,1.1125,0.8191
2,DO concentration,17,Seasonal linear,0.691,0.9863,0.7047
3,DO concentration,17,Gradient boosting,0.637,1.0679,0.8062
4,DO concentration,17,Quadratic ridge,0.632,1.0763,0.8103
5,Water temperature,24,Linear chemistry,-0.067,7.3236,6.0950
6,Water temperature,24,Linear + Laplacian,-0.058,7.2894,6.0487
7,Water temperature,24,Seasonal linear,0.678,4.0220,3.0478
8,Water temperature,24,Gradient boosting,-0.157,7.6254,6.3672
9,Water temperature,24,Quadratic ridge,-0.065,7.3147,6.0877


In [37]:
r2_table = (
    model_loo_results
    .pivot(
        index=[
            "Target",
            "Nodes"
        ],
        columns="Model",
        values="R2"
    )
    .reset_index()
)


r2_table.columns.name = None


target_order = [
    "DO concentration",
    "Water temperature",
    "pH",
    "Specific conductance",
    "Turbidity"
]


r2_table["Target"] = pd.Categorical(
    r2_table["Target"],
    categories=target_order,
    ordered=True
)


r2_table = (
    r2_table
    .sort_values(
        "Target"
    )
    .reset_index(
        drop=True
    )
)


display(
    r2_table.style.format({

        "Linear chemistry":
            "{:.3f}",

        "Linear + Laplacian":
            "{:.3f}",

        "Seasonal linear":
            "{:.3f}",

        "Gradient boosting":
            "{:.3f}",

        "Quadratic ridge":
            "{:.3f}"
    })
)

,Target,Nodes,Gradient boosting,Linear + Laplacian,Linear chemistry,Quadratic ridge,Seasonal linear
0,DO concentration,17,0.637,0.607,0.621,0.632,0.691
1,Water temperature,24,-0.157,-0.058,-0.067,-0.065,0.678
2,pH,19,-0.066,0.074,-0.044,-0.041,0.016
3,Specific conductance,24,-0.046,0.106,-0.104,-0.058,-0.079
4,Turbidity,17,-0.081,-0.036,0.007,-0.004,0.055


In [38]:
model_loo_results.to_csv(
    "all_models_node_loo_results.csv",
    index=False
)


model_loo_predictions.to_csv(
    "all_models_node_loo_predictions.csv",
    index=False
)


model_loo_tuning.to_csv(
    "all_models_node_loo_tuning.csv",
    index=False
)

In [39]:
def build_full_edge_weight_lookup(
    edge_features,
    scheme
):

    lookup = {}


    for row in edge_features.itertuples():

        score = (

            scheme["area"]
            * row.z_log_area

            +

            scheme["width"]
            * row.z_log_width

            +

            scheme["discharge"]
            * row.z_log_discharge

            +

            scheme["slope"]
            * row.z_log_slope
        )


        score = np.clip(
            score,
            -5.0,
            5.0
        )


        distance_km = max(
            float(row.distance_km),
            1e-6
        )


        w = (
            np.exp(score)
            /
            distance_km
        )


        lookup[
            frozenset(
                [
                    row.u,
                    row.v
                ]
            )
        ] = w


    return lookup

In [40]:
def build_compressed_weighted_L(
    G_master,
    edge_weight_lookup
):

    node_order = list(
        G_master.nodes()
    )


    node_index = {
        node: i
        for i, node
        in enumerate(node_order)
    }


    rows = []
    cols = []
    vals = []


    for u, v, data in G_master.edges(
        data=True
    ):

        path = data.get(
            "original_path",
            None
        )


        if path is None:

            path = [
                u,
                v
            ]


        resistances = []


        for a, b in zip(
            path[:-1],
            path[1:]
        ):

            key = frozenset(
                [
                    a,
                    b
                ]
            )


            if key not in edge_weight_lookup:

                raise KeyError(
                    f"Missing full edge {a} -- {b}"
                )


            w = edge_weight_lookup[
                key
            ]


            resistances.append(
                1.0 / w
            )


        effective_weight = (
            1.0
            /
            np.sum(
                resistances
            )
        )


        i = node_index[u]
        j = node_index[v]


        rows.extend(
            [
                i,
                j
            ]
        )

        cols.extend(
            [
                j,
                i
            ]
        )

        vals.extend(
            [
                effective_weight,
                effective_weight
            ]
        )


    W = sparse.coo_matrix(
        (
            vals,
            (
                rows,
                cols
            )
        ),
        shape=(
            len(node_order),
            len(node_order)
        )
    ).tocsr()


    degree = np.asarray(
        W.sum(
            axis=1
        )
    ).ravel()


    L = (
        sparse.diags(
            degree
        )
        -
        W
    ).tocsr()


    return (
        node_order,
        node_index,
        L
    )

In [41]:
def kron_reduce_compressed(
    L,
    node_order,
    node_index
):

    boundary_idx = np.array(
        [
            node_index[node]
            for node in boundary_nodes
        ],
        dtype=int
    )


    all_idx = np.arange(
        len(node_order)
    )


    interior_mask = np.ones(
        len(node_order),
        dtype=bool
    )


    interior_mask[
        boundary_idx
    ] = False


    interior_idx = all_idx[
        interior_mask
    ]


    L_BB = (
        L[
            boundary_idx,
            :
        ][
            :,
            boundary_idx
        ]
        .tocsc()
    )


    L_BI = (
        L[
            boundary_idx,
            :
        ][
            :,
            interior_idx
        ]
        .tocsc()
    )


    L_IB = (
        L[
            interior_idx,
            :
        ][
            :,
            boundary_idx
        ]
        .tocsc()
    )


    L_II = (
        L[
            interior_idx,
            :
        ][
            :,
            interior_idx
        ]
        .tocsc()
    )


    lu = splu(
        L_II
    )


    solution = lu.solve(
        L_IB.toarray()
    )


    L_eff = (
        L_BB.toarray()
        -
        L_BI @ solution
    )


    L_eff = np.asarray(
        L_eff
    )


    L_eff = (
        L_eff
        +
        L_eff.T
    ) / 2.0


    for i in range(
        L_eff.shape[0]
    ):

        L_eff[i, i] = -(
            L_eff[i, :].sum()
            -
            L_eff[i, i]
        )


    diag = np.diag(
        L_eff
    )


    scale = (
        diag[
            diag > 1e-12
        ].mean()
    )


    return (
        L_eff
        /
        scale
    )

In [42]:
compressed_effective_laplacians = {}


for name, scheme in WEIGHT_SCHEMES.items():

    if name not in effective_laplacians_clean:
        continue


    print(
        "Building compressed:",
        name
    )


    weight_lookup = (
        build_full_edge_weight_lookup(
            edge_features,
            scheme
        )
    )


    (
        comp_nodes,
        comp_index,
        L_comp
    ) = build_compressed_weighted_L(
        G_master,
        weight_lookup
    )


    L_eff_comp = kron_reduce_compressed(
        L_comp,
        comp_nodes,
        comp_index
    )


    compressed_effective_laplacians[
        name
    ] = L_eff_comp

Building compressed: distance_only
Building compressed: area
Building compressed: width
Building compressed: slope
Building compressed: area_width


In [43]:
comparison_rows = []


for name in effective_laplacians_clean:

    L_full_eff = (
        effective_laplacians_clean[
            name
        ]
    )


    L_comp_eff = (
        compressed_effective_laplacians[
            name
        ]
    )


    diff = (
        L_full_eff
        -
        L_comp_eff
    )


    relative_error = (
        np.linalg.norm(
            diff,
            ord="fro"
        )
        /
        np.linalg.norm(
            L_full_eff,
            ord="fro"
        )
    )


    comparison_rows.append({

        "Weighting":
            name,

        "Max Absolute Difference":
            np.max(
                np.abs(
                    diff
                )
            ),

        "Relative Frobenius Difference":
            relative_error
    })


laplacian_comparison = pd.DataFrame(
    comparison_rows
)


display(
    laplacian_comparison
)

,Weighting,Max Absolute Difference,Relative Frobenius Difference
0,distance_only,2.808420e-12,5.658121e-13
1,area,1.156741e-12,2.606058e-13
2,width,2.003009e-12,5.412655e-13
3,slope,1.030376e-11,2.445991e-12
4,area_width,2.746525e-12,6.117096e-13


In [44]:
import numpy as np

longitudes = np.array([
    float(data["longitude"])
    for _, data in G_full.nodes(data=True)
])

latitudes = np.array([
    float(data["latitude"])
    for _, data in G_full.nodes(data=True)
])


min_lon = longitudes.min()
max_lon = longitudes.max()

min_lat = latitudes.min()
max_lat = latitudes.max()


center_lon = (min_lon + max_lon) / 2
center_lat = (min_lat + max_lat) / 2


print("River Network Geographic Coverage")
print("=" * 45)

print(f"Minimum longitude: {min_lon:.6f}")
print(f"Maximum longitude: {max_lon:.6f}")

print(f"Minimum latitude:  {min_lat:.6f}")
print(f"Maximum latitude:  {max_lat:.6f}")

print()

print(
    "Bounding box:"
)

print(
    f"[{min_lon:.6f}, "
    f"{min_lat:.6f}, "
    f"{max_lon:.6f}, "
    f"{max_lat:.6f}]"
)

print()

print(
    "Center coordinate:"
)

print(
    f"Latitude:  {center_lat:.6f}"
)

print(
    f"Longitude: {center_lon:.6f}"
)

River Network Geographic Coverage
Minimum longitude: -115.201667
Maximum longitude: -107.060833
Minimum latitude:  31.817500
Maximum latitude:  42.157500

Bounding box:
[-115.201667, 31.817500, -107.060833, 42.157500]

Center coordinate:
Latitude:  36.987500
Longitude: -111.131250


In [45]:
import numpy as np
import pandas as pd


colorado_seasonal = (
    node_hourly_all.copy()
)


colorado_seasonal["time"] = pd.to_datetime(
    colorado_seasonal["time"],
    utc=True
)


colorado_seasonal["hour"] = (
    colorado_seasonal["time"].dt.hour
    +
    colorado_seasonal["time"].dt.minute / 60.0
)


colorado_seasonal["day_of_year"] = (
    colorado_seasonal["time"]
    .dt.dayofyear
)


colorado_seasonal["hour_sin"] = np.sin(
    2
    * np.pi
    * colorado_seasonal["hour"]
    / 24.0
)


colorado_seasonal["hour_cos"] = np.cos(
    2
    * np.pi
    * colorado_seasonal["hour"]
    / 24.0
)


colorado_seasonal["doy_sin"] = np.sin(
    2
    * np.pi
    * colorado_seasonal["day_of_year"]
    / 365.25
)


colorado_seasonal["doy_cos"] = np.cos(
    2
    * np.pi
    * colorado_seasonal["day_of_year"]
    / 365.25
)


TIME_FEATURES = [
    "hour_sin",
    "hour_cos",
    "doy_sin",
    "doy_cos"
]


print(
    "Rows:",
    len(colorado_seasonal)
)

print(
    "Graph nodes:",
    colorado_seasonal[
        "node"
    ].nunique()
)

Rows: 225339
Graph nodes: 26


In [46]:
TARGET_CONFIG = {

    "DO concentration": [
        "Water temperature",
        "Specific conductance"
    ],

    "Water temperature": [
        "Specific conductance"
    ],

    "pH": [
        "Specific conductance"
    ],

    "Specific conductance": [
        "Water temperature"
    ],

    "Turbidity": [
        "Specific conductance",
        "pH"
    ]
}

In [47]:
SEASONAL_TARGET_CONFIG = {}


for target, chemistry_features in TARGET_CONFIG.items():

    SEASONAL_TARGET_CONFIG[
        target
    ] = (
        chemistry_features
        +
        TIME_FEATURES
    )


for target, features in SEASONAL_TARGET_CONFIG.items():

    print(
        target,
        "->",
        features
    )

DO concentration -> ['Water temperature', 'Specific conductance', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']
Water temperature -> ['Specific conductance', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']
pH -> ['Specific conductance', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']
Specific conductance -> ['Water temperature', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']
Turbidity -> ['Specific conductance', 'pH', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']


In [48]:
colorado_seasonal_datasets = {}


for target, features in SEASONAL_TARGET_CONFIG.items():

    df = (
        colorado_seasonal[
            [
                "node",
                "time",
                target
            ]
            +
            features
        ]
        .dropna(
            subset=[
                target
            ]
            +
            features
        )
        .copy()
    )


    colorado_seasonal_datasets[
        target
    ] = df


    print(
        "\n",
        target
    )

    print(
        "  nodes:",
        df["node"].nunique()
    )

    print(
        "  node-hours:",
        len(df)
    )


 DO concentration
  nodes: 17
  node-hours: 114607

 Water temperature
  nodes: 24
  node-hours: 144980

 pH
  nodes: 19
  node-hours: 121011

 Specific conductance
  nodes: 24
  node-hours: 144980

 Turbidity
  nodes: 17
  node-hours: 88095


In [49]:
colorado_seasonal_results = []

colorado_seasonal_predictions = {}

colorado_seasonal_tuning = {}


for target, features in SEASONAL_TARGET_CONFIG.items():

    print(
        "\n"
        + "=" * 75
    )

    print(
        "TARGET:",
        target
    )

    print(
        "Predictors:",
        features
    )

    print(
        "=" * 75
    )


    result, predictions, tuning = (
        nested_loo_hydrologic(

            df=
                colorado_seasonal_datasets[
                    target
                ],

            target=
                target,

            features=
                features
        )
    )


    colorado_seasonal_results.append(
        result
    )


    colorado_seasonal_predictions[
        target
    ] = predictions


    colorado_seasonal_tuning[
        target
    ] = tuning


TARGET: DO concentration
Predictors: ['Water temperature', 'Specific conductance', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']

TARGET: DO concentration
Predictors: ['Water temperature', 'Specific conductance', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']
LOO nodes: 17
  Outer fold 1/17 - node 67739434
  Outer fold 2/17 - node 81497938
  Outer fold 3/17 - node 82982643
  Outer fold 4/17 - node 83027786
  Outer fold 5/17 - node 83222751
  Outer fold 6/17 - node 84197086
  Outer fold 7/17 - node 88786999
  Outer fold 8/17 - node 90706215
  Outer fold 9/17 - node 92116891
  Outer fold 10/17 - node 93300040
  Outer fold 11/17 - node 93392084
  Outer fold 12/17 - node 96496279
  Outer fold 13/17 - node 97532807
  Outer fold 14/17 - node 126377584
  Outer fold 15/17 - node 131520862
  Outer fold 16/17 - node 186559442
  Outer fold 17/17 - node 203177183

TARGET: Water temperature
Predictors: ['Specific conductance', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']

TARGET: Water temperature

In [50]:
colorado_seasonal_results = pd.DataFrame(
    colorado_seasonal_results
)


target_order = [
    "DO concentration",
    "Water temperature",
    "pH",
    "Specific conductance",
    "Turbidity"
]


colorado_seasonal_results["Target"] = pd.Categorical(

    colorado_seasonal_results["Target"],

    categories=
        target_order,

    ordered=True
)


colorado_seasonal_results = (
    colorado_seasonal_results
    .sort_values(
        "Target"
    )
    .reset_index(
        drop=True
    )
)


display(
    colorado_seasonal_results.style.format({

        "Linear R2":
            "{:.3f}",

        "Graph R2":
            "{:.3f}",

        "Linear RMSE":
            "{:.4f}",

        "Graph RMSE":
            "{:.4f}",

        "Graph RMSE Improvement %":
            "{:.2f}%"
    })
)

,Target,Nodes,Linear R2,Graph R2,Linear RMSE,Graph RMSE,Graph RMSE Improvement %
0,DO concentration,17,0.691,0.683,0.9863,0.9984,-1.23%
1,Water temperature,24,0.678,0.739,4.0219,3.6191,10.02%
2,pH,19,0.016,0.130,0.2350,0.2210,5.95%
3,Specific conductance,24,-0.079,0.146,460.8822,409.9114,11.06%
4,Turbidity,17,0.055,-0.008,550.0134,568.2948,-3.32%


In [51]:
original = (
    hydro_results[
        [
            "Target",
            "Nodes",
            "Linear R2",
            "Graph R2",
            "Linear RMSE",
            "Graph RMSE"
        ]
    ]
    .copy()
)


original = original.rename(
    columns={

        "Linear R2":
            "Linear original R2",

        "Graph R2":
            "Graph original R2",

        "Linear RMSE":
            "Linear original RMSE",

        "Graph RMSE":
            "Graph original RMSE"
    }
)


seasonal = (
    colorado_seasonal_results[
        [
            "Target",
            "Nodes",
            "Linear R2",
            "Graph R2",
            "Linear RMSE",
            "Graph RMSE"
        ]
    ]
    .copy()
)


seasonal = seasonal.rename(
    columns={

        "Linear R2":
            "Linear seasonal R2",

        "Graph R2":
            "Graph seasonal R2",

        "Linear RMSE":
            "Linear seasonal RMSE",

        "Graph RMSE":
            "Graph seasonal RMSE"
    }
)


colorado_seasonal_comparison = (
    original
    .merge(
        seasonal,
        on=[
            "Target",
            "Nodes"
        ],
        how="inner"
    )
)


colorado_seasonal_comparison[
    "Linear R2 improvement"
] = (
    colorado_seasonal_comparison[
        "Linear seasonal R2"
    ]
    -
    colorado_seasonal_comparison[
        "Linear original R2"
    ]
)


colorado_seasonal_comparison[
    "Graph R2 improvement"
] = (
    colorado_seasonal_comparison[
        "Graph seasonal R2"
    ]
    -
    colorado_seasonal_comparison[
        "Graph original R2"
    ]
)


display(
    colorado_seasonal_comparison[
        [
            "Target",
            "Nodes",
            "Linear original R2",
            "Linear seasonal R2",
            "Linear R2 improvement",
            "Graph original R2",
            "Graph seasonal R2",
            "Graph R2 improvement"
        ]
    ]
    .style
    .format({

        "Linear original R2":
            "{:.3f}",

        "Linear seasonal R2":
            "{:.3f}",

        "Linear R2 improvement":
            "{:+.3f}",

        "Graph original R2":
            "{:.3f}",

        "Graph seasonal R2":
            "{:.3f}",

        "Graph R2 improvement":
            "{:+.3f}"
    })
)

,Target,Nodes,Linear original R2,Linear seasonal R2,Linear R2 improvement,Graph original R2,Graph seasonal R2,Graph R2 improvement
0,DO concentration,17,0.621,0.691,+0.070,0.607,0.683,+0.077
1,Water temperature,24,-0.056,0.678,+0.734,-0.058,0.739,+0.797
2,pH,19,-0.035,0.016,+0.052,0.074,0.130,+0.056
3,Specific conductance,24,-0.089,-0.079,+0.010,0.106,0.146,+0.040
4,Turbidity,17,0.023,0.055,+0.033,-0.036,-0.008,+0.028


In [52]:
scheme_rows = []


for target, tuning in colorado_seasonal_tuning.items():

    counts = (
        tuning[
            "graph_scheme"
        ]
        .value_counts()
    )


    for scheme, count in counts.items():

        scheme_rows.append({

            "Target":
                target,

            "Weighting":
                scheme,

            "Selected folds":
                count,

            "Total folds":
                len(tuning),

            "Selection %":
                100
                * count
                / len(tuning)
        })


seasonal_scheme_selection = (
    pd.DataFrame(
        scheme_rows
    )
    .sort_values(
        [
            "Target",
            "Selected folds"
        ],
        ascending=[
            True,
            False
        ]
    )
)


display(
    seasonal_scheme_selection.style.format({
        "Selection %":
            "{:.1f}%"
    })
)

,Target,Weighting,Selected folds,Total folds,Selection %
0,DO concentration,slope,7,17,41.2%
1,DO concentration,width,6,17,35.3%
2,DO concentration,area_width,3,17,17.6%
3,DO concentration,distance_only,1,17,5.9%
6,Specific conductance,area_width,22,24,91.7%
7,Specific conductance,area,2,24,8.3%
8,Turbidity,area_width,11,17,64.7%
9,Turbidity,width,4,17,23.5%
10,Turbidity,area,2,17,11.8%
4,Water temperature,area,24,24,100.0%
